# 🔍 BrandPulse — Complete Pipeline

**Run cells in order: 1 → 2 → 3 → 4 → 5 → 6 → 7 → 8**

| Cell | What it does |
|---|---|
| 1 | Install all libraries |
| 2 | Set secrets & env vars |
| 3 | Test MongoDB connection |
| 4 | **Scrape** Twitter, Reddit, YouTube, Google Maps → MongoDB |
| 5 | **Analyse** with DeepSeek AI (aspects, sentiment, suggestions) |
| 6 | **Save** analysis results to MongoDB (feeds the dashboard) |
| 7 | **QA** — validate everything worked |

> ⚠️ Cells 4–6 must run in the **same session** so variables carry over.

## Cell 1 — Install Libraries

In [ ]:
!pip install "pymongo[srv]" dnspython certifi python-dotenv requests praw youtube-comment-downloader tqdm langdetect
!pip install --upgrade youtube-transcript-api
!pip install -U openai-whisper faster-whisper
!pip install -U yt-dlp

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 18.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 189.3/189.3 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.7/318.7 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 42.9 MB/s eta 0:00:00
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=5489976499df44274bd577ae1def607317751d81d5abfb9f539cbb349b425471
  Stored in directory: /root/.cache/pip/wheels/c1/67/88/e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 11.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 12.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing 

## Cell 2 — Set Secrets & Environment Variables

> Add `MONGO_URI`, `BRAND`, `TW_BEARER_TOKEN` etc. to **Colab Secrets** (key icon in sidebar) so you never paste them again.

In [ ]:
import os

# ---------- Mongo ----------
os.environ["MONGO_URI"] = "YOUR_MONGO_URI_HERE"

# ---------- Twitter ----------
os.environ["TW_BEARER_TOKEN"] = "YOUR_TWITTER_BEARER_TOKEN"

# ---------- Reddit ----------
os.environ["REDDIT_CLIENT_ID"] = "YOUR_REDDIT_CLIENT_ID"
os.environ["REDDIT_CLIENT_SECRET"] = "YOUR_REDDIT_CLIENT_SECRET"
os.environ["REDDIT_USER_AGENT"] = "YOUR_REDDIT_USER_AGENT"
os.environ["REDDIT_USERNAME"] = "YOUR_REDDIT_USERNAME"
os.environ["REDDIT_PASSWORD"] = "YOUR_REDDIT_PASSWORD"

# ---------- SerpAPI (Google Reviews) ----------
os.environ["SERPAPI_KEY"] = "YOUR_SERPAPI_KEY"

# ---------- Brand (set ONCE here) ----------
os.environ["BRAND"] = "KFC"   # change this when you want another brand

print("✅ Secrets & BRAND set in env.")
print("BRAND =", os.environ["BRAND"])


✅ Secrets & BRAND set in env.
BRAND = KFC


## Cell 3 — Test MongoDB Connection

In [ ]:
# Cell 4 – Quick Mongo ping test using env vars

from pymongo import MongoClient
import certifi, os

uri = os.getenv("MONGO_URI")
if not uri:
    raise ValueError("MONGO_URI not set. Run the secrets cell (Cell 2) first.")

client = MongoClient(uri, tls=True, tlsCAFile=certifi.where(), serverSelectionTimeoutMS=30000)
client.admin.command("ping")
test_db = client[os.getenv("MONGO_DB_NAME", "brandpulse")]
print("✅ Mongo ping OK, DB:", test_db.name)

✅ Mongo ping OK, DB: brandpulse


## Cell 4 — Scrape Data (Twitter + Reddit + YouTube + Google Maps)

Collects last 120 days of brand mentions and stores in MongoDB `records` collection.

In [ ]:
# -*- coding: utf-8 -*-
"""
BrandPulse -> MongoDB Atlas (no CSV)
FIXED VERSION - Key changes:
  1. YouTube: fixed language filter (don't drop videos with no captions)
  2. YouTube: fixed threading (comments after meta, not concurrent)
  3. YouTube: fixed transcript API detection for newer versions
  4. YouTube: fixed audio/video cleanup (only after successful transcription)
  5. MongoDB: handle empty source_id collisions
  6. General: safer error handling throughout
"""

import os
import re
import time
import json
import shutil
import datetime
import subprocess
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests
import praw
from tqdm import tqdm
from youtube_comment_downloader import YoutubeCommentDownloader

# --- YouTube transcripts: safe import (handles all versions) ---
try:
    from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled, NoTranscriptFound
    # Newer versions (>=0.6.0) use a different API style; detect both
    try:
        # Try new API style first (>=0.6.0)
        _test_fetcher = YouTubeTranscriptApi()
        HAS_YT_TRANSCRIPT_API = True
        YT_NEW_API = True
    except Exception:
        # Fall back to class-method style
        HAS_YT_TRANSCRIPT_API = hasattr(YouTubeTranscriptApi, "get_transcript")
        YT_NEW_API = False
except Exception:
    YouTubeTranscriptApi = None
    TranscriptsDisabled = Exception
    NoTranscriptFound = Exception
    HAS_YT_TRANSCRIPT_API = False
    YT_NEW_API = False

from pymongo import MongoClient, ASCENDING
import gridfs
from bson import ObjectId
import mimetypes

from langdetect import detect, LangDetectException

# ================== CONFIG ==================
CONFIG = {
    "brand": "KFC",

    "secrets": {
        "twitter_bearer_token": os.getenv("TW_BEARER_TOKEN", ""),
        "reddit_client_id":     os.getenv("REDDIT_CLIENT_ID", ""),
        "reddit_client_secret": os.getenv("REDDIT_CLIENT_SECRET", ""),
        "reddit_user_agent":    os.getenv("REDDIT_USER_AGENT", ""),
        "reddit_username":      os.getenv("REDDIT_USERNAME", ""),
        "reddit_password":      os.getenv("REDDIT_PASSWORD", ""),
        "serpapi_key":          os.getenv("SERPAPI_KEY", ""),
    },

    "mongo": {
        "uri": os.getenv("MONGO_URI", ""),
        "db_name": os.getenv("MONGO_DB_NAME", "brandpulse"),
        "records_coll": "records",
        "delete_local_after_upload": True
    },

    "out_dirs": {
        "root": "data",
        "twitter_media": "data/twitter",
        "reddit_media":  "data/reddit",
        "youtube_video": "data/youtube/videos",
        "youtube_audio": "data/youtube/audio",
        "youtube_thumbs": "data/youtube/thumbs",
        "transcripts": "data/transcripts",
        "google_media": "data/google",
        "frames": "data/frames"
    },

    "twitter": {
        "search_url": "https://api.twitter.com/2/tweets/search/recent",
        "languages": ["en", "si"],
        "tweet_fields": "created_at,author_id,lang,source,attachments,public_metrics",
        "expansions": "attachments.media_keys,author_id",
        "user_fields": "name,username",
        "media_fields": "type,url,preview_image_url,variants,alt_text,width,height",
        "max_results_per_request": 100
    },

    "reddit": {
        "subreddits": ["srilanka", "SL", "LK", "CasualConversation", "AskReddit"],
        "days_back": 120,
    },

    "youtube": {
        "languages": ["en", "si"],
        "download_media": True,
        # FIX: reduced workers to avoid file collision
        "max_workers": 4,
        "days_back": 120,
        # FIX: don't drop videos just because language can't be detected
        "strict_lang_filter": False,
    },

    "google": {
        "country": "Sri Lanka",
        "city_center_ll": "6.9271,79.8612,11z",
        "max_places": None,
        "max_reviews_per_place": 200,
        "days_back": 120,
    },

    "limits": {
        "reddit_submissions_per_sub": 10,
        "reddit_comments_per_post": 200,
        "youtube_videos_per_query": 10,
        "youtube_comments_per_video": 200,
        "days_back": 120,
    },

    "analysis": {
        "use_youtube_captions_first": True,
        "use_whisper_fallback": True,
        "whisper_model": "base"
    },

    "flags": {
        "reddit_filter_comment_contains_brand": True
    }
}

# ============= MONGO HELPERS =============
_mongo_client = None
_db = None
_fs = None

def mongo_connect():
    global _mongo_client, _db, _fs
    import certifi
    _mongo_client = MongoClient(
        CONFIG["mongo"]["uri"],
        tls=True,
        tlsCAFile=certifi.where(),
        serverSelectionTimeoutMS=30000
    )
    _db = _mongo_client[CONFIG["mongo"]["db_name"]]
    _fs = gridfs.GridFS(_db)
    # Unique index – only on non-empty source_id
    _db[CONFIG["mongo"]["records_coll"]].create_index(
        [("platform", ASCENDING), ("record_type", ASCENDING), ("source_id", ASCENDING)],
        unique=True,
        sparse=True,   # FIX: sparse=True so empty/null source_id rows don't collide
        name="uniq_platform_type_source"
    )
    return _db, _fs

def db_size_gb():
    try:
        stats = _db.command("dbstats")
        return (stats.get("dataSize", 0) + stats.get("indexSize", 0)) / (1024**3)
    except Exception:
        return 0.0

def gridfs_put(path, metadata=None):
    metadata = metadata or {}
    try:
        with open(path, "rb") as f:
            oid = _fs.put(f, filename=os.path.basename(path), **{"metadata": metadata})
        if CONFIG["mongo"]["delete_local_after_upload"]:
            try:
                os.remove(path)
            except Exception:
                pass
        return oid
    except Exception as e:
        print(f"[GridFS] Failed to store {path}: {e}")
        return None

def guess_kind_from_suffix(path):
    suf = Path(path).suffix.lower()
    if suf in {".jpg", ".jpeg", ".png", ".gif", ".webp"}:
        return "image"
    if suf in {".mp4", ".mov", ".m4v", ".webm"}:
        return "video"
    if suf in {".mp3", ".wav", ".m4a"}:
        return "audio"
    if suf in {".txt", ".vtt", ".srt"}:
        return "text"
    return "file"

def upload_media_list(paths, source, extra_meta=None):
    out = []
    for p in paths or []:
        if not p or not Path(p).exists():
            continue
        kind = guess_kind_from_suffix(p)
        mime = mimetypes.guess_type(p)[0] or "application/octet-stream"
        meta = {"source": source, "kind": kind}
        if extra_meta:
            meta.update(extra_meta)
        oid = gridfs_put(p, metadata=meta)
        if oid:
            out.append({
                "gridfs_id": str(oid),
                "filename": os.path.basename(p),
                "content_type": mime,
                "kind": kind
            })
    return out

def db_upsert_record(doc):
    coll = _db[CONFIG["mongo"]["records_coll"]]
    # FIX: if source_id is empty, generate a unique one to avoid index collision
    if not doc.get("source_id"):
        doc["source_id"] = f"auto_{doc.get('platform','x')}_{int(time.time()*1000)}"
    filt = {
        "platform": doc.get("platform"),
        "record_type": doc.get("record_type"),
        "source_id": doc.get("source_id")
    }
    doc["ingested_at"] = datetime.datetime.utcnow()
    try:
        coll.update_one(filt, {"$set": doc}, upsert=True)
    except Exception as e:
        print(f"[Mongo] Upsert error: {e}")

def delete_brand_records(brand):
    coll = _db[CONFIG["mongo"]["records_coll"]]
    result = coll.delete_many({"brand": brand})
    print(f"[Mongo] Deleted {result.deleted_count} old documents for brand '{brand}'.")

# ============= GENERIC HELPERS =============
def ensure_dirs(*paths):
    for p in paths:
        Path(p).mkdir(parents=True, exist_ok=True)

def to_iso(dt):
    import datetime as _dt
    if isinstance(dt, (int, float)):
        return _dt.datetime.utcfromtimestamp(dt).isoformat() + "Z"
    if isinstance(dt, _dt.datetime):
        if dt.tzinfo is None:
            return dt.isoformat() + "Z"
        return dt.astimezone(_dt.timezone.utc).isoformat().replace("+00:00", "Z")
    if isinstance(dt, str):
        return dt
    return ""

def safe_filename(name: str) -> str:
    return re.sub(r"[^\w\-.]+", "_", name.strip())[:180]

ALLOWED_LANGS = ["en", "si"]

def looks_like_sinhala(text: str) -> bool:
    for ch in text:
        if "\u0D80" <= ch <= "\u0DFF":
            return True
    return False

def is_allowed_lang(text: str, allowed_langs=None) -> bool:
    if not text or not text.strip():
        return False
    if allowed_langs is None:
        allowed_langs = ALLOWED_LANGS
    if looks_like_sinhala(text):
        return True
    try:
        lang = detect(text)
        return lang in allowed_langs
    except LangDetectException:
        return False
    except Exception:
        return False

def run_cmd(cmd):
    return subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

def download_url(url, dest_path, session=None, max_tries=3):
    sess = session or requests.Session()
    for i in range(max_tries):
        try:
            with sess.get(url, stream=True, timeout=60) as r:
                if r.status_code != 200:
                    continue
                with open(dest_path, "wb") as f:
                    for chunk in r.iter_content(1024 * 256):
                        if chunk:
                            f.write(chunk)
            return True
        except Exception as e:
            if i == max_tries - 1:
                print(f"[Download] Failed {url}: {e}")
    return False

def yt_dlp_download(media_url, out_template):
    cmd = ["yt-dlp", media_url, "-o", out_template, "--no-playlist", "--no-warnings", "--quiet"]
    subprocess.run(cmd, check=False)

# ============= TRANSCRIPTION =============
def whisper_transcribe(audio_or_video_path, model_name="base", out_txt=""):
    text = ""
    out_txt = out_txt or (str(Path(audio_or_video_path).with_suffix(".txt")))
    try:
        try:
            from faster_whisper import WhisperModel
            model = WhisperModel(model_name)
            segments, _ = model.transcribe(audio_or_video_path)
            text = "\n".join(
                [s.text.strip() for s in segments if getattr(s, "text", "").strip()]
            )
        except Exception:
            import whisper
            model = whisper.load_model(model_name)
            result = model.transcribe(audio_or_video_path)
            text = (result.get("text") or "").strip()
    except Exception as e:
        print(f"[Whisper] Failed: {e}")
        return "", ""
    try:
        with open(out_txt, "w", encoding="utf-8") as f:
            f.write(text)
    except Exception:
        pass
    return text, out_txt

# ============= BRAND QUERIES =============
def build_queries_from_brand(brand):
    brand_clean = re.sub(r"\s+", "", brand)
    brand_hashtag = "#" + brand_clean

    twitter_terms = [
        brand,
        f"{brand} review",
        f"{brand} complaints",
        f"{brand} SL",
        f"{brand} SriLanka",
        brand_hashtag,
        f"{brand_hashtag}review",
        f"{brand_hashtag}reviews",
        f"{brand_hashtag}complaints",
        f"{brand_hashtag}SriLanka",
        f"#{brand_clean}SriLanka",
    ]

    sl_words = [
        '"Sri Lanka"', "SriLanka", "SL", "Colombo", "Kandy",
        "Galle", "Negombo", "Kurunegala", "Jaffna", "Gampaha",
    ]

    reddit_query = f'"{brand}" OR "{brand_clean}"'

    yt_queries = [
        f"{brand} review Sri Lanka",
        f"{brand} experience Sri Lanka",
        f"{brand} Sri Lanka",
        f"{brand_clean} Sri Lanka",
        f"{brand} Colombo",
        f"{brand} Kandy",
    ]

    return twitter_terms, reddit_query, yt_queries, sl_words

# ============= TWITTER =============
def tw_headers(bearer_token: str):
    return {"Authorization": f"Bearer {bearer_token}", "User-Agent": "BrandPulseTwitterScraper"}

def tw_build_query(terms, languages, sl_words):
    brand_part = " OR ".join(terms) if terms else ""
    lang_part = " OR ".join([f"lang:{l}" for l in languages]) if languages else ""
    sl_part = " OR ".join(sl_words) if sl_words else ""
    clauses = []
    if brand_part:
        clauses.append(f"({brand_part})")
    if sl_part:
        clauses.append(f"({sl_part})")
    if lang_part:
        clauses.append(f"({lang_part})")
    base = " ".join(clauses).strip()
    return f"{base} -is:retweet" if base else "-is:retweet"

def tw_connect(url, headers, params):
    consecutive_429 = 0
    while True:
        try:
            r = requests.get(url, headers=headers, params=params, timeout=30)
            if r.status_code == 429:
                consecutive_429 += 1
                if consecutive_429 >= 2:
                    print("[Twitter] Two consecutive 429s. Stopping.")
                    return None
                reset_time = int(r.headers.get("x-rate-limit-reset", time.time() + 60))
                wait_s = max(0, reset_time - time.time() + 5)
                print(f"[Twitter] Rate limited. Sleeping {wait_s:.1f}s")
                time.sleep(wait_s)
                continue
            if r.status_code != 200:
                print(f"[Twitter] Error: {r.status_code} {r.text}")
                return None
            return r.json()
        except requests.exceptions.RequestException as e:
            print(f"[Twitter] Network error: {e}")
            return None

def best_video_variant(variants):
    best = None
    best_br = -1
    for v in variants or []:
        if v.get("content_type") == "video/mp4":
            br = v.get("bit_rate", 0) or 0
            if br > best_br and v.get("url"):
                best = v["url"]
                best_br = br
    return best

def download_twitter_media(media_entities, dest_dir):
    ensure_dirs(dest_dir)
    saved = []
    session = requests.Session()
    for m in media_entities or []:
        mtype = m.get("type")
        if mtype == "photo" and m.get("url"):
            url = m["url"]
            fname = safe_filename(Path(url).name)
            path = os.path.join(dest_dir, fname)
            if download_url(url, path, session=session):
                saved.append(path)
        elif mtype in ("video", "animated_gif"):
            vurl = best_video_variant(m.get("variants"))
            if vurl:
                fname = safe_filename(Path(vurl).name.split("?")[0])
                path = os.path.join(
                    dest_dir,
                    fname if fname.endswith(".mp4") else fname + ".mp4"
                )
                if download_url(vurl, path, session=session):
                    saved.append(path)
    return saved

def run_twitter(config, secrets, brand, out_dirs):
    bearer = secrets["twitter_bearer_token"]
    if not bearer:
        print("[Twitter] Skipped: missing Bearer token.")
        return

    twitter_terms, _, _, sl_words = build_queries_from_brand(brand)
    headers = tw_headers(bearer)
    query = tw_build_query(twitter_terms, config["languages"], sl_words)

    next_token = None
    page = 0
    media_lookup = {}

    print("[Twitter] Start fetching...")
    while True:
        params = {
            "query": query,
            "tweet.fields": config["tweet_fields"],
            "expansions": config["expansions"],
            "media.fields": config["media_fields"],
            "user.fields": config["user_fields"],
            "max_results": config["max_results_per_request"]
        }
        if next_token:
            params["next_token"] = next_token

        data = tw_connect(config["search_url"], headers, params)
        if data is None:
            break

        includes = data.get("includes", {})
        media_list = includes.get("media", [])
        users = {u["id"]: u for u in includes.get("users", [])}
        media_lookup.update({m.get("media_key"): m for m in media_list})

        if "data" in data:
            batch = data["data"]
            print(f"[Twitter] Page {page + 1}: {len(batch)} tweets")
            for t in batch:
                if t.get("lang") not in ("en", "si"):
                    continue

                tweet_id = t.get("id", "")
                author_obj = users.get(t.get("author_id", ""), {})
                author_name = author_obj.get("username") or author_obj.get("name", "")

                grid_media = []
                transcript_text = ""
                transcript_gridfs_id = None

                mkeys = (t.get("attachments") or {}).get("media_keys", [])
                tweet_media = [media_lookup.get(k) for k in mkeys if media_lookup.get(k)]
                local_media = download_twitter_media(
                    tweet_media, out_dirs["twitter_media"]
                ) if tweet_media else []

                video_cands = [p for p in local_media if Path(p).suffix.lower() in (".mp4", ".mov", ".m4v", ".webm")]
                image_cands = [p for p in local_media if Path(p).suffix.lower() in (".jpg", ".jpeg", ".png", ".gif", ".webp")]
                audio_cands = [p for p in local_media if Path(p).suffix.lower() in (".mp3", ".wav", ".m4a")]

                grid_media.extend(upload_media_list(image_cands, "twitter"))

                target_audio = audio_cands[0] if audio_cands else (video_cands[0] if video_cands else None)
                if target_audio and CONFIG["analysis"]["use_whisper_fallback"]:
                    text, txt_path = whisper_transcribe(
                        target_audio,
                        CONFIG["analysis"]["whisper_model"],
                        out_txt=os.path.join(out_dirs["transcripts"], f"twitter_{tweet_id}.txt")
                    )
                    transcript_text = text
                    if txt_path:
                        tid = gridfs_put(txt_path, metadata={"source": "twitter", "kind": "text", "role": "transcript"})
                        transcript_gridfs_id = str(tid) if tid else None

                if CONFIG["mongo"]["delete_local_after_upload"]:
                    for p in video_cands + audio_cands:
                        if p and Path(p).exists():
                            try:
                                os.remove(p)
                            except Exception:
                                pass

                metrics = t.get("public_metrics", {}) or {}
                doc = {
                    "platform": "twitter",
                    "record_type": "post",
                    "brand": brand,
                    "text": t.get("text", ""),
                    "author": author_name,
                    "created_at": to_iso(t.get("created_at", "")),
                    "source_title": "",
                    "source_id": tweet_id,
                    "url": f"https://twitter.com/i/web/status/{tweet_id}" if tweet_id else "",
                    "lang": t.get("lang", ""),
                    "subreddit": "",
                    "score": "",
                    "likes": metrics.get("like_count", ""),
                    "video_id": "",
                    "video_title": "",
                    "channel": "",
                    "upload_date": "",
                    "duration": "",
                    "view_count": "",
                    "media_files": "",
                    "captions_text": "",
                    "transcript_text": transcript_text,
                    "transcript_gridfs_id": transcript_gridfs_id,
                    "media": grid_media,
                    "frames": []
                }
                db_upsert_record(doc)

            page += 1
            next_token = data.get("meta", {}).get("next_token")
            if next_token:
                time.sleep(1)
                continue
        break

# ============= REDDIT =============
def download_reddit_media_from_url(post_url, out_dir):
    ensure_dirs(out_dir)
    out_template = os.path.join(out_dir, "%(id)s.%(ext)s")
    pre_files = set(str(p) for p in Path(out_dir).glob("*"))
    yt_dlp_download(post_url, out_template)
    return [str(p) for p in Path(out_dir).glob("*") if str(p) not in pre_files]

def get_reddit_post_images(submission, out_dir):
    ensure_dirs(out_dir)
    paths = []

    url = (submission.url or "").strip()
    if url and re.search(r"\.(jpg|jpeg|png|gif|webp)(\?.*)?$", url, flags=re.I):
        fname = safe_filename(Path(url).name)
        dest = os.path.join(out_dir, fname)
        if download_url(url, dest):
            paths.append(dest)

    if getattr(submission, "is_gallery", False):
        media = getattr(submission, "media_metadata", {}) or {}
        for key, meta in media.items():
            s = meta.get("s") or {}
            img_url = s.get("u") or s.get("gif")
            if not img_url:
                continue
            img_url = img_url.replace("&amp;", "&")
            fname = safe_filename(Path(img_url).name or f"{submission.id}_{key}.jpg")
            dest = os.path.join(out_dir, fname)
            if download_url(img_url, dest):
                paths.append(dest)

    if not paths:
        try:
            more_files = download_reddit_media_from_url(
                f"https://www.reddit.com{submission.permalink}", out_dir
            )
            paths.extend(more_files)
        except Exception as e:
            print(f"[Reddit] media download failed: {e}")

    return paths

def run_reddit(config, secrets, brand, out_dirs):
    need = [secrets["reddit_client_id"], secrets["reddit_client_secret"], secrets["reddit_password"]]
    if any((not x or str(x).startswith("YOUR_")) for x in need):
        print("[Reddit] Skipped: missing credentials.")
        return

    reddit = praw.Reddit(
        client_id=secrets["reddit_client_id"],
        client_secret=secrets["reddit_client_secret"],
        user_agent=secrets["reddit_user_agent"],
        username=secrets["reddit_username"],
        password=secrets["reddit_password"],
        check_for_async=False,
    )

    _, reddit_query, _, _ = build_queries_from_brand(brand)

    print("[Reddit] Start fetching...")
    for sub_name in config["subreddits"]:
        sub = reddit.subreddit(sub_name)
        try:
            submissions = sub.search(reddit_query, sort="new", limit=CONFIG["limits"]["reddit_submissions_per_sub"])
            for submission in submissions:
                text_for_lang = ((submission.title or "") + " " + (submission.selftext or "")).strip()
                if not is_allowed_lang(text_for_lang):
                    continue

                media_files = get_reddit_post_images(submission, out_dirs["reddit_media"])

                grid_media = []
                transcript_text = ""
                transcript_gridfs_id = None

                imgs = [p for p in media_files if Path(p).suffix.lower() in (".jpg", ".jpeg", ".png", ".gif", ".webp")]
                vids = [p for p in media_files if Path(p).suffix.lower() in (".mp4", ".mov", ".m4v", ".webm")]
                auds = [p for p in media_files if Path(p).suffix.lower() in (".mp3", ".wav", ".m4a")]

                grid_media.extend(upload_media_list(imgs, "reddit"))

                target_audio = auds[0] if auds else (vids[0] if vids else None)
                if target_audio and CONFIG["analysis"]["use_whisper_fallback"]:
                    text, txt_path = whisper_transcribe(
                        target_audio,
                        CONFIG["analysis"]["whisper_model"],
                        out_txt=os.path.join(out_dirs["transcripts"], f"reddit_{submission.id}.txt")
                    )
                    transcript_text = text
                    if txt_path:
                        tid = gridfs_put(txt_path, metadata={"source": "reddit", "kind": "text", "role": "transcript"})
                        transcript_gridfs_id = str(tid) if tid else None

                if CONFIG["mongo"]["delete_local_after_upload"]:
                    for p in vids + auds:
                        if p and Path(p).exists():
                            try:
                                os.remove(p)
                            except Exception:
                                pass

                doc_post = {
                    "platform": "reddit",
                    "record_type": "post",
                    "brand": brand,
                    "text": (submission.selftext or "").strip(),
                    "author": getattr(submission.author, "name", "") if submission.author else "",
                    "created_at": to_iso(submission.created_utc),
                    "source_title": submission.title or "",
                    "source_id": submission.id,
                    "url": f"https://www.reddit.com{submission.permalink}",
                    "lang": "",
                    "subreddit": sub_name,
                    "score": "", "likes": "", "video_id": "", "video_title": "",
                    "channel": "", "upload_date": "", "duration": "", "view_count": "",
                    "media_files": "", "captions_text": "",
                    "transcript_text": transcript_text,
                    "transcript_gridfs_id": transcript_gridfs_id,
                    "media": grid_media,
                    "frames": []
                }
                db_upsert_record(doc_post)

                submission.comments.replace_more(limit=2)
                taken = 0
                for c in submission.comments.list():
                    if taken >= CONFIG["limits"]["reddit_comments_per_post"]:
                        break
                    text = (getattr(c, "body", "") or "").strip()
                    if not text:
                        continue
                    if CONFIG["flags"]["reddit_filter_comment_contains_brand"] and brand.lower() not in text.lower():
                        continue
                    if not is_allowed_lang(text):
                        continue

                    comment_image_paths = []
                    img_urls = re.findall(r"(https?://\S+\.(?:jpg|jpeg|png|gif|webp))", text, flags=re.IGNORECASE)
                    for idx_img, url_img in enumerate(img_urls):
                        ensure_dirs(out_dirs["reddit_media"])
                        fname = safe_filename(Path(url_img).name or f"{c.id}_{idx_img}.jpg")
                        dest = os.path.join(out_dirs["reddit_media"], fname)
                        if download_url(url_img, dest):
                            comment_image_paths.append(dest)

                    comment_media = upload_media_list(
                        comment_image_paths, "reddit",
                        extra_meta={"role": "comment_image", "comment_id": getattr(c, "id", "")}
                    )

                    doc_comment = {
                        "platform": "reddit",
                        "record_type": "comment",
                        "brand": brand,
                        "text": text,
                        "author": getattr(c.author, "name", "") if getattr(c, "author", None) else "",
                        "created_at": to_iso(getattr(c, "created_utc", None)),
                        "source_title": submission.title or "",
                        "source_id": getattr(c, "id", ""),
                        "url": f"https://www.reddit.com{getattr(c, 'permalink', '')}",
                        "lang": "", "subreddit": sub_name,
                        "score": "", "likes": "", "video_id": "", "video_title": "",
                        "channel": "", "upload_date": "", "duration": "", "view_count": "",
                        "media_files": "", "captions_text": "",
                        "transcript_text": "", "transcript_gridfs_id": None,
                        "media": comment_media, "frames": []
                    }
                    db_upsert_record(doc_comment)
                    taken += 1
                time.sleep(0.25)

        except Exception as e:
            print(f"[Reddit] Error in r/{sub_name}: {e}")

# ============= YOUTUBE =============
def yt_search_videos(queries, max_per_query):
    found = []
    seen_ids = set()
    for q in queries:
        cmd = ["yt-dlp", f"ytsearch{max_per_query}:{q}", "--dump-json", "--skip-download"]
        print(f"[YouTube] Searching: {q}")
        proc = run_cmd(cmd)
        if proc.returncode != 0:
            print(f"[YouTube] yt-dlp error: {proc.stderr[:200]}")
        lines = proc.stdout.strip().splitlines()
        for line in lines:
            try:
                item = json.loads(line)
                vid_id = item.get("id", "")
                if not vid_id or vid_id in seen_ids:
                    continue
                seen_ids.add(vid_id)

                thumb = item.get("thumbnail", "")
                if not thumb:
                    thumbs = item.get("thumbnails") or []
                    if thumbs and isinstance(thumbs, list):
                        # prefer highest resolution thumbnail
                        best = max(thumbs, key=lambda t: t.get("width", 0) if isinstance(t, dict) else 0)
                        thumb = best.get("url", "") if isinstance(best, dict) else ""

                found.append({
                    "video_id": vid_id,
                    "title": item.get("title", ""),
                    "channel": item.get("channel", "") or item.get("uploader", ""),
                    "upload_date": item.get("upload_date", ""),
                    "duration": item.get("duration", ""),
                    "view_count": item.get("view_count", ""),
                    "thumbnail": thumb,
                    "description": (item.get("description") or "")[:500],
                })
            except Exception as e:
                print(f"[YouTube] Parse error: {e}")
    print(f"[YouTube] Found {len(found)} unique videos total")
    return found


# FIX: Rewritten to handle both old and new youtube_transcript_api versions
def get_youtube_captions_text(video_id, langs):
    if not HAS_YT_TRANSCRIPT_API:
        return ""
    try:
        if YT_NEW_API:
            # New API (>=0.6.0): instantiate and use list_transcripts
            ytt = YouTubeTranscriptApi()
            transcript_list = ytt.list(video_id)
            # Try requested languages first
            for lang in langs:
                try:
                    t = transcript_list.find_transcript([lang])
                    segments = t.fetch()
                    return " ".join(seg.get("text", "") for seg in segments if seg.get("text"))
                except Exception:
                    continue
            # Fallback: any transcript
            try:
                t = next(iter(transcript_list))
                segments = t.fetch()
                return " ".join(seg.get("text", "") for seg in segments if seg.get("text"))
            except Exception:
                return ""
        else:
            # Old API: class-method style
            try:
                transcript = YouTubeTranscriptApi.get_transcript(video_id, languages=langs)
                return " ".join(seg.get("text", "") for seg in transcript if seg.get("text"))
            except (TranscriptsDisabled, NoTranscriptFound):
                pass
            try:
                transcript = YouTubeTranscriptApi.get_transcript(video_id)
                return " ".join(seg.get("text", "") for seg in transcript if seg.get("text"))
            except Exception:
                return ""
    except Exception as e:
        print(f"[YouTube] Captions error for {video_id}: {e}")
        return ""


def yt_download_audio(video_id, audio_dir):
    """Download only audio (MP3) for transcription. Returns path or ''."""
    ensure_dirs(audio_dir)
    url = f"https://www.youtube.com/watch?v={video_id}"
    out_audio = os.path.join(audio_dir, f"{video_id}.%(ext)s")
    cmd_audio = [
        "yt-dlp", url,
        "-x", "--audio-format", "mp3",
        "-o", out_audio,
        "--no-playlist", "--no-warnings", "--quiet"
    ]
    result = subprocess.run(cmd_audio, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"[YouTube] Audio download failed for {video_id}: {result.stderr[:200]}")

    audio_path = os.path.join(audio_dir, f"{video_id}.mp3")
    if Path(audio_path).exists():
        return audio_path
    # Sometimes ext differs
    matches = list(Path(audio_dir).glob(f"{video_id}.*"))
    return str(matches[0]) if matches else ""


def process_video(v, out_dirs, brand):
    """Download thumbnail, get captions/transcript, store video metadata."""
    video_id = v["video_id"]
    url = f"https://www.youtube.com/watch?v={video_id}"

    captions_text = ""
    transcript_text = ""
    transcript_gridfs_id = None
    media_meta = []

    # 1) Thumbnail -> GridFS
    thumb_url = v.get("thumbnail", "")
    if thumb_url:
        ensure_dirs(out_dirs["youtube_thumbs"])
        ext = Path(thumb_url.split("?")[0]).suffix or ".jpg"
        fname = f"{video_id}_thumb{ext}"
        dest = os.path.join(out_dirs["youtube_thumbs"], fname)
        if download_url(thumb_url, dest):
            media_meta.extend(
                upload_media_list([dest], "youtube", extra_meta={"role": "thumbnail", "video_id": video_id})
            )

    # 2) Captions
    if CONFIG["analysis"]["use_youtube_captions_first"]:
        captions_text = get_youtube_captions_text(video_id, CONFIG["youtube"]["languages"])
        if captions_text:
            print(f"[YouTube] Got captions for {video_id} ({len(captions_text)} chars)")

    # 3) Whisper fallback (only if no captions)
    if CONFIG["analysis"]["use_whisper_fallback"] and not captions_text:
        audio_file = yt_download_audio(video_id, out_dirs["youtube_audio"])
        if audio_file:
            print(f"[YouTube] Transcribing {video_id} with Whisper...")
            text, txt_path = whisper_transcribe(
                audio_file,
                CONFIG["analysis"]["whisper_model"],
                out_txt=os.path.join(out_dirs["transcripts"], f"youtube_{video_id}.txt")
            )
            transcript_text = text
            # FIX: only delete audio AFTER transcription is done
            if txt_path and Path(txt_path).exists():
                tid = gridfs_put(txt_path, metadata={"source": "youtube", "kind": "text", "role": "transcript"})
                transcript_gridfs_id = str(tid) if tid else None
            if CONFIG["mongo"]["delete_local_after_upload"] and audio_file and Path(audio_file).exists():
                try:
                    os.remove(audio_file)
                except Exception:
                    pass

    # FIX: Language filter - only apply if strict mode AND we have enough text to check
    # Don't drop videos just because title is short or ambiguous
    if CONFIG["youtube"].get("strict_lang_filter", False):
        lang_source = captions_text or transcript_text or v.get("title", "") or v.get("description", "")
        if lang_source and not is_allowed_lang(lang_source):
            print(f"[YouTube] Skipping video {video_id}: language not allowed")
            return None

    doc = {
        "platform": "youtube",
        "record_type": "video",
        "brand": brand,
        "text": v.get("description", ""),
        "author": v.get("channel", ""),
        "created_at": v.get("upload_date", ""),
        "source_title": v.get("title", ""),
        "source_id": video_id,
        "url": url,
        "lang": "",
        "subreddit": "", "score": "", "likes": "",
        "video_id": video_id,
        "video_title": v.get("title", ""),
        "channel": v.get("channel", ""),
        "upload_date": v.get("upload_date", ""),
        "duration": v.get("duration", ""),
        "view_count": v.get("view_count", ""),
        "media_files": "",
        "captions_text": captions_text,
        "transcript_text": transcript_text,
        "transcript_gridfs_id": transcript_gridfs_id,
        "media": media_meta,
        "frames": []
    }

    db_upsert_record(doc)
    return video_id


def scrape_video_comments(v, brand):
    """Scrape comments for a YouTube video."""
    downloader = YoutubeCommentDownloader()
    url = f"https://www.youtube.com/watch?v={v['video_id']}"
    out = []
    try:
        for i, c in enumerate(downloader.get_comments_from_url(url)):
            if i >= CONFIG["limits"]["youtube_comments_per_video"]:
                break
            text = c.get("text", "") or ""
            if not is_allowed_lang(text):
                continue
            out.append({
                "platform": "youtube",
                "record_type": "comment",
                "brand": brand,
                "text": text,
                "author": c.get("author", "N/A"),
                "created_at": c.get("time", ""),
                "source_title": v["title"],
                "source_id": c.get("cid", ""),
                "url": url,
                "lang": "", "subreddit": "", "score": "",
                "likes": c.get("votes", 0),
                "video_id": v["video_id"],
                "video_title": v["title"],
                "channel": v.get("channel", ""),
                "upload_date": v.get("upload_date", ""),
                "duration": v.get("duration", ""),
                "view_count": "",
                "media_files": "", "captions_text": "",
                "transcript_text": "", "transcript_gridfs_id": None,
                "media": [], "frames": []
            })
    except Exception as e:
        print(f"[YouTube] Comment error for {url}: {e}")
    return out


def run_youtube(config, brand, out_dirs):
    _, _, yt_queries, _ = build_queries_from_brand(brand)
    videos = yt_search_videos(yt_queries, CONFIG["limits"]["youtube_videos_per_query"])

    if not videos:
        print("[YouTube] No videos found.")
        return

    # FIX: Process video metadata first (sequential within thread pool),
    # then scrape comments. Avoids file conflicts and ensures DB records exist first.
    print(f"[YouTube] Processing {len(videos)} videos (metadata + transcription)...")
    with ThreadPoolExecutor(max_workers=config.get("max_workers", 4)) as ex:
        futures = {ex.submit(process_video, v, out_dirs, brand): v for v in videos}
        for fut in as_completed(futures):
            try:
                result = fut.result()
                if result:
                    print(f"[YouTube] Stored video: {result}")
            except Exception as e:
                print(f"[YouTube] process_video error: {e}")

    print(f"[YouTube] Scraping comments for {len(videos)} videos...")
    with ThreadPoolExecutor(max_workers=config.get("max_workers", 4)) as ex:
        futures = {ex.submit(scrape_video_comments, v, brand): v for v in videos}
        for fut in as_completed(futures):
            try:
                comments = fut.result()
                for doc in comments:
                    db_upsert_record(doc)
            except Exception as e:
                print(f"[YouTube] comment scrape error: {e}")

    print(f"[YouTube] Done.")

# ============= GOOGLE MAPS REVIEWS =============
def run_google_reviews(google_config, secrets, brand):
    serpapi_key = os.getenv("SERPAPI_KEY") or secrets.get("serpapi_key", "")
    if not serpapi_key:
        print("[Google] Skipped: SERPAPI_KEY not set.")
        return

    base_url = "https://serpapi.com/search"
    country = google_config.get("country", "Sri Lanka")
    max_places = google_config.get("max_places", None)
    max_reviews_per_place = google_config.get("max_reviews_per_place", 200)
    days_back = google_config.get("days_back", 120)
    cutoff_dt = datetime.datetime.now(datetime.timezone.utc) - datetime.timedelta(days=days_back)

    search_params = {
        "engine": "google_maps",
        "type": "search",
        "q": f"{brand} Sri Lanka",
        "google_domain": "google.lk",
        "hl": "en",
        "gl": "lk",
        "api_key": serpapi_key,
    }

    print(f"[Google] Searching places for '{brand}' in {country}...")
    try:
        resp = requests.get(base_url, params=search_params, timeout=60)
        resp.raise_for_status()
        data = resp.json()
    except Exception as e:
        print(f"[Google] Place search failed: {e}")
        return

    places = []
    local_results = data.get("local_results") or []
    if isinstance(local_results, list):
        places.extend(local_results)
    place_results = data.get("place_results")
    if isinstance(place_results, dict):
        places.append(place_results)

    if not places:
        print("[Google] No places found.")
        return
    if max_places is not None:
        places = places[:max_places]

    print(f"[Google] Found {len(places)} places.")

    for pi, place in enumerate(places, start=1):
        title = place.get("title", "")
        address = place.get("address", "")
        data_id = place.get("data_id")
        place_id = place.get("place_id")
        reviews_link = place.get("reviews_link", "") or place.get("link", "")
        gps = place.get("gps_coordinates") or {}
        lat, lng = gps.get("latitude"), gps.get("longitude")

        if not (data_id or place_id):
            print(f"[Google] Skip place without data_id/place_id: {title}")
            continue

        print(f"[Google] ({pi}/{len(places)}) {title} | {address}")

        num_fetched = 0
        next_page_token = None
        stop_fetching = False

        while not stop_fetching:
            review_params = {
                "api_key": serpapi_key,
                "engine": "google_maps_reviews",
                "hl": "en",
                "sort_by": "newestFirst",
            }
            if data_id:
                review_params["data_id"] = data_id
            else:
                review_params["place_id"] = place_id
            if next_page_token:
                review_params["next_page_token"] = next_page_token

            try:
                r = requests.get(base_url, params=review_params, timeout=60)
                r.raise_for_status()
                rdata = r.json()
            except Exception as e:
                print(f"  [Google] Reviews request failed: {e}")
                break

            reviews = rdata.get("reviews") or []
            if not reviews:
                break

            for rv in reviews:
                iso_date = rv.get("iso_date") or rv.get("date")
                created_iso = ""
                created_dt = None
                if iso_date:
                    iso_str = str(iso_date).replace("Z", "+00:00")
                    for fmt in ("%Y-%m-%dT%H:%M:%S%z", "%Y-%m-%d", "%Y-%m-%d %H:%M:%S%z"):
                        try:
                            created_dt = datetime.datetime.strptime(iso_str, fmt)
                            break
                        except Exception:
                            continue
                    if created_dt:
                        if created_dt.tzinfo is None:
                            created_dt = created_dt.replace(tzinfo=datetime.timezone.utc)
                        else:
                            created_dt = created_dt.astimezone(datetime.timezone.utc)
                        created_iso = created_dt.isoformat().replace("+00:00", "Z")

                if created_dt and created_dt < cutoff_dt:
                    stop_fetching = True
                    break

                text = (
                    rv.get("snippet")
                    or (rv.get("extracted_snippet") or {}).get("original")
                    or rv.get("comment")
                    or ""
                )

                if not is_allowed_lang(text):
                    continue

                author_info = rv.get("author") or {}
                author_name = author_info.get("name") or rv.get("user") or ""
                rating = rv.get("rating")
                review_id = rv.get("review_id") or rv.get("id") or ""

                review_image_paths = []
                for idx_img, img in enumerate(rv.get("images") or []):
                    url_img = img if isinstance(img, str) else (
                        img.get("thumbnail") or img.get("image") or img.get("url")
                        if isinstance(img, dict) else None
                    )
                    if not url_img:
                        continue
                    ensure_dirs(CONFIG["out_dirs"]["google_media"])
                    fname = safe_filename(Path(url_img).name or f"{review_id}_{idx_img}.jpg")
                    dest = os.path.join(CONFIG["out_dirs"]["google_media"], fname)
                    if download_url(url_img, dest):
                        review_image_paths.append(dest)

                grid_media = upload_media_list(
                    review_image_paths, "google_maps",
                    extra_meta={"place_title": title, "review_id": review_id}
                )

                doc = {
                    "platform": "google_maps",
                    "record_type": "review",
                    "brand": brand,
                    "text": text,
                    "author": author_name,
                    "created_at": created_iso,
                    "source_title": title,
                    "source_id": review_id,
                    "url": reviews_link,
                    "lang": "en",
                    "subreddit": "", "score": "", "likes": "",
                    "video_id": "", "video_title": "", "channel": "",
                    "upload_date": "", "duration": "", "view_count": "",
                    "rating": rating,
                    "location_address": address,
                    "location_latitude": lat,
                    "location_longitude": lng,
                    "media_files": "", "captions_text": "",
                    "transcript_text": "", "transcript_gridfs_id": None,
                    "media": grid_media, "frames": []
                }
                db_upsert_record(doc)
                num_fetched += 1

                if num_fetched >= max_reviews_per_place:
                    stop_fetching = True
                    break

            pagination = rdata.get("serpapi_pagination") or {}
            next_page_token = pagination.get("next_page_token")
            if not next_page_token:
                break

        print(f"  [Google] Stored {num_fetched} reviews for {title}")

# ============= MAIN =============
if __name__ == "__main__":
    print("=== BrandPulse → MongoDB Atlas (FIXED) ===")

    out_dirs = CONFIG["out_dirs"]
    ensure_dirs(*out_dirs.values())

    brand = (os.getenv("BRAND") or CONFIG["brand"]).strip()
    CONFIG["brand"] = brand
    print(f"[RUN] Brand: {brand}")

    db, fs = mongo_connect()
    print(f"[Mongo] Connected to '{CONFIG['mongo']['db_name']}'. Size ~ {db_size_gb():.3f} GB")

    delete_brand_records(brand)

    secrets = CONFIG["secrets"]

    print("\n--- Twitter ---")
    run_twitter(CONFIG["twitter"], secrets, brand, out_dirs)

    print("\n--- Reddit ---")
    run_reddit(CONFIG["reddit"], secrets, brand, out_dirs)

    print("\n--- YouTube ---")
    run_youtube(CONFIG["youtube"], brand, out_dirs)

    print("\n--- Google Maps ---")
    run_google_reviews(CONFIG["google"], secrets, brand)

    print("\n✅ Done. All records stored in MongoDB.")


=== BrandPulse → MongoDB Atlas (FIXED) ===
[RUN] Brand: KFC
[Mongo] Connected to 'brandpulse'. Size ~ 0.022 GB
[Mongo] Deleted 1654 old documents for brand 'KFC'.

--- Twitter ---
[Twitter] Start fetching...
[Twitter] Error: 403 {"client_id":"31174174","detail":"When authenticating requests to the Twitter API v2 endpoints, you must use keys and tokens from a Twitter developer App that is attached to a Project. You can create a project via the developer portal.","registration_url":"https://developer.twitter.com/en/docs/projects/overview","title":"Client Forbidden","required_enrollment":"Appropriate Level of API Access","reason":"client-not-enrolled","type":"https://api.twitter.com/2/problems/client-forbidden"}

--- Reddit ---
[Reddit] Start fetching...


/tmp/ipykernel_214/3289501110.py:248: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  return _dt.datetime.utcfromtimestamp(dt).isoformat() + "Z"
/tmp/ipykernel_214/3289501110.py:229: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  doc["ingested_at"] = datetime.datetime.utcnow()
/tmp/ipykernel_214/3289501110.py:248: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  return _dt.datetime.utcfromtimestamp(dt).isoformat() + "Z"
/tmp/ipykernel_214/3289501110.py:229: Depre

[Reddit] Error in r/SL: received 403 HTTP response
[Reddit] Error in r/LK: received 404 HTTP response


/tmp/ipykernel_214/3289501110.py:248: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  return _dt.datetime.utcfromtimestamp(dt).isoformat() + "Z"
/tmp/ipykernel_214/3289501110.py:229: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  doc["ingested_at"] = datetime.datetime.utcnow()
/tmp/ipykernel_214/3289501110.py:248: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  return _dt.datetime.utcfromtimestamp(dt).isoformat() + "Z"
/tmp/ipykernel_214/3289501110.py:229: Depre


--- YouTube ---
[YouTube] Searching: KFC review Sri Lanka
[YouTube] Searching: KFC experience Sri Lanka
[YouTube] Searching: KFC Sri Lanka
[YouTube] Searching: KFC Sri Lanka
[YouTube] Searching: KFC Colombo
[YouTube] Searching: KFC Kandy
[YouTube] yt-dlp error: WARNING: [youtube] No supported JavaScript runtime could be found. Only deno is enabled by default; to use another runtime add  --js-runtimes RUNTIME[:PATH]  to your command/config. YouTube extraction
[YouTube] Found 39 unique videos total
[YouTube] Processing 39 videos (metadata + transcription)...
[YouTube] Transcribing wEbJ1dtjCWs with Whisper...
[YouTube] Transcribing Wlw8t0tPs0o with Whisper...
[YouTube] Transcribing QktqUitaIOk with Whisper...
[YouTube] Transcribing fvtLSEEiEXY with Whisper...


...

[YouTube] Stored video: T1ZKQ_zn0nk
[YouTube] Captions error for GvOjJAuGt_I: 
Could not retrieve a transcript for the video https://www.youtube.com/watch?v=GvOjJAuGt_I! This is most likely caused by:

Subtitles are disabled for this video

If you are sure that the described cause is not responsible for this error and that a transcript should be retrievable, please create an issue at https://github.com/jdepoix/youtube-transcript-api/issues. Please add which version of youtube_transcript_api you are using and provide the information needed to replicate the error. Also make sure that there are no open issues which already describe your problem!
[YouTube] Transcribing GvOjJAuGt_I with Whisper...


/tmp/ipykernel_214/3289501110.py:229: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  doc["ingested_at"] = datetime.datetime.utcnow()


[YouTube] Stored video: GvOjJAuGt_I
[YouTube] Transcribing 8wFziloRMWs with Whisper...


/tmp/ipykernel_214/3289501110.py:229: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  doc["ingested_at"] = datetime.datetime.utcnow()


[YouTube] Stored video: 8wFziloRMWs
[YouTube] Stored video: QMJFmrOZuzA
[YouTube] Stored video: GUImROmrdok
[YouTube] Stored video: PkSIGVL5D2k
[YouTube] Scraping comments for 39 videos...
[YouTube] Done.

--- Google Maps ---
[Google] Searching places for 'KFC' in Sri Lanka...
[Google] Found 20 places.
[Google] (1/20) KFC - Union Place | 278 Dr Colvin R de Silva Mawatha, Colombo
  [Google] Stored 14 reviews for KFC - Union Place
[Google] (2/20) KFC - Majestic City | 10 Station Rd, Colombo 00400
  [Google] Stored 8 reviews for KFC - Majestic City
[Google] (3/20) KFC - Colombo 1 | 23 Canal Row, Colombo 00100
  [Google] Stored 8 reviews for KFC - Colombo 1
[Google] (4/20) KFC - Kotahena | WVX5+CQV, George R. De Silva Mawatha, Colombo 01300
  [Google] Stored 9 reviews for KFC - Kotahena
[Google] (5/20) KFC - Wellawatta | 29 Charlemont Rd, Colombo 00600
  [Google] Stored 4 reviews for KFC - Wellawatta
[Google] (6/20) Sweta Food Corner | No.06 Medamanuwara ඌරුගලKandy, 20940
  [Google] Stored

## Cell 5 — XLM-RoBERTa + DeepSeek AI Analysis

Loads records from MongoDB, runs **XLM-RoBERTa** (multilingual — supports Sinhala + English) for fast sentiment pre-filtering, then **DeepSeek** for deep aspect-based analysis.

**Key improvements over previous version:**
- ✅ VADER replaced with `cardiffnlp/twitter-xlm-roberta-base-sentiment` — works on Sinhala text
- ✅ YouTube titles are passed as **background context only** — DeepSeek now judges sentiment from the comment text, not the title
- ✅ Brand field added to all saved aspect/suggestion rows

Produces `df_aspects`, `df_suggestions`, and `final_summary` → saved to MongoDB by Cell 6.


In [ ]:
# =======================================================================
# BrandPulse — AI Analysis  (XLM-RoBERTa + DeepSeek) — FIXED v3
# Changes vs previous version:
#   1. VADER replaced with XLM-RoBERTa (supports Sinhala + English)
#   2. YouTube: title passed separately so DeepSeek weights it ONCE,
#      then analyses each comment on its own merit
#   3. analysis_text now contains only comments/body text; title is
#      injected into the prompt as context, not body content
# =======================================================================

!pip -q install requests tqdm pandas pymongo certifi transformers torch sentencepiece

import pandas as pd
import requests
import json
import os
import re
import time
import getpass
from tqdm.auto import tqdm
from pymongo import MongoClient
import certifi
import torch
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification

tqdm.pandas()

# =======================================================================
# STEP 1: LOAD SECRETS
# =======================================================================
def safe_str(val):
    if val is None: return ""
    if isinstance(val, dict):
        return str(list(val.values())[0]).strip() if val else ""
    return str(val).strip()

def safe_getpass(prompt):
    try:
        val = getpass.getpass(prompt)
        return safe_str(val)
    except Exception as e:
        print(f"  getpass error: {e}")
        return input(prompt)

def load_secrets():
    secrets = {}
    try:
        from google.colab import userdata
        for name in ["deepseekAPI", "de", "deepseek", "DEEPSEEK_API_KEY"]:
            try:
                val = safe_str(userdata.get(name))
                if val:
                    secrets["deepseek_key"] = val
                    print(f"  DeepSeek key loaded from secret '{name}'")
                    break
            except Exception as e:
                print(f"  Could not read secret '{name}': {e}")

        for name in ["MONGO_URI", "mongo_uri", "MongoURI"]:
            try:
                val = safe_str(userdata.get(name))
                if val:
                    secrets["mongo_uri"] = val
                    print(f"  MongoDB URI loaded from secret '{name}'")
                    break
            except Exception as e:
                print(f"  Could not read secret '{name}': {e}")

        secrets["db_name"] = "brandpulse"
        for name in ["MONGO_DB_NAME", "mongo_db"]:
            try:
                val = safe_str(userdata.get(name))
                if val:
                    secrets["db_name"] = val
                    break
            except Exception:
                pass

        for name in ["BRAND", "brand"]:
            try:
                val = safe_str(userdata.get(name))
                if val:
                    secrets["brand"] = val
                    print(f"  Brand: {val}")
                    break
            except Exception:
                pass

    except Exception as e:
        print(f"  Colab Secrets unavailable: {e}")

    if not secrets.get("deepseek_key"):
        print("\n  DeepSeek key missing. Paste below:")
        secrets["deepseek_key"] = safe_getpass("  DeepSeek API key: ")
    if not secrets.get("mongo_uri"):
        print("\n  MongoDB URI missing. Paste below:")
        secrets["mongo_uri"] = safe_getpass("  MongoDB URI: ")
    if not secrets.get("brand"):
        secrets["brand"] = input("\n  Brand name [KFC]: ").strip() or "KFC"

    return secrets

if not globals().get("SECRETS_LOADED"):
    print("Loading secrets...")
    _S = load_secrets()
    DEEPSEEK_API_KEY = _S.get("deepseek_key", "")
    MONGO_URI        = _S.get("mongo_uri", "")
    MONGO_DB_NAME    = _S.get("db_name", "brandpulse")
    BRAND            = _S.get("brand", "KFC")
    SECRETS_LOADED   = True

    if not DEEPSEEK_API_KEY:
        raise ValueError("DeepSeek API key is still empty. Check Colab Secrets.")
    if not MONGO_URI:
        raise ValueError("MongoDB URI is still empty. Check Colab Secrets.")

    print(f"\nReady. Brand='{BRAND}' | Key starts with: '{DEEPSEEK_API_KEY[:8]}...'")
else:
    print(f"Secrets already loaded. Brand='{BRAND}'")

DEEPSEEK_URL = "https://api.deepseek.com/v1/chat/completions"

# =======================================================================
# STEP 2: CONNECT TO MONGODB + LOAD RECORDS
# =======================================================================
print(f"\nConnecting to MongoDB...")
_client = MongoClient(MONGO_URI, tls=True, tlsCAFile=certifi.where(), serverSelectionTimeoutMS=30000)
_client.admin.command("ping")
_db   = _client[MONGO_DB_NAME]
_coll = _db["records"]

cursor = _coll.find(
    {"brand": BRAND},
    {"platform": 1, "record_type": 1, "brand": 1, "url": 1,
     "created_at": 1, "source_title": 1, "text": 1,
     "captions_text": 1, "transcript_text": 1, "rating": 1}
)

df = pd.DataFrame(list(cursor))
if df.empty:
    raise ValueError(f"No documents found for brand='{BRAND}'. Run the scraper first.")

df["_id"]   = df["_id"].astype(str)
df["brand"] = BRAND

for col in ["source_title", "text", "captions_text", "transcript_text",
            "platform", "record_type", "url", "created_at", "rating"]:
    if col not in df.columns: df[col] = ""
    df[col] = df[col].fillna("")

print(f"Loaded {len(df)} records for brand '{BRAND}'")
print(df.groupby(["platform", "record_type"]).size().to_string())

# =======================================================================
# STEP 3: BUILD analysis_text
# NOTE: For YouTube, we keep source_title SEPARATE from analysis_text
#       so the prompt can inject it as context (not repeated per comment)
# =======================================================================
def clean_text(s) -> str:
    return re.sub(r"\s+", " ", str(s or "")).strip()

def make_analysis_text(row) -> str:
    """
    For YouTube records: analysis_text = comments/captions/transcript ONLY.
    Title is stored separately and injected once into the DeepSeek prompt.
    For all other platforms: combine everything as before.
    """
    platform = str(row.get("platform", "")).lower()
    if platform == "youtube":
        # Exclude source_title here — it will be passed as 'video_title' to DeepSeek
        parts = [
            clean_text(row.get("text", "")),
            clean_text(row.get("captions_text", "")),
            clean_text(row.get("transcript_text", "")),
        ]
    else:
        parts = [
            clean_text(row.get("source_title", "")),
            clean_text(row.get("text", "")),
            clean_text(row.get("captions_text", "")),
            clean_text(row.get("transcript_text", "")),
        ]
    return clean_text(" ".join(p for p in parts if p))

df["analysis_text"] = df.apply(make_analysis_text, axis=1)
df_usable = df[df["analysis_text"].str.len() >= 15].copy()
print(f"\nUsable rows: {len(df_usable)} / {len(df)}")

if df_usable.empty:
    raise ValueError("No usable text. Check the scraper ran successfully.")

# =======================================================================
# STEP 4: XLM-RoBERTa MULTILINGUAL SENTIMENT
# (Replaces VADER — works for Sinhala, English and many other languages)
# =======================================================================
print("\nLoading XLM-RoBERTa multilingual sentiment model...")
print("(First run downloads ~1.1 GB — subsequent runs use cache)")

XLM_MODEL_NAME = "cardiffnlp/twitter-xlm-roberta-base-sentiment"

_xlm_tokenizer = AutoTokenizer.from_pretrained(XLM_MODEL_NAME)
_xlm_model     = AutoModelForSequenceClassification.from_pretrained(XLM_MODEL_NAME)
_xlm_pipe      = pipeline(
    "sentiment-analysis",
    model=_xlm_model,
    tokenizer=_xlm_tokenizer,
    truncation=True,
    max_length=512,
    device=0 if torch.cuda.is_available() else -1
)
# Label map: the cardiffnlp model outputs Negative/Neutral/Positive
_XLM_LABEL_MAP = {"negative": "negative", "neutral": "neutral", "positive": "positive"}

def xlm_label(text: str) -> str:
    """Get sentiment label using XLM-RoBERTa. Handles Sinhala + English."""
    if not text or len(text.strip()) < 10:
        return "neutral"
    try:
        result = _xlm_pipe(text[:512])[0]
        label  = result["label"].lower()
        return _XLM_LABEL_MAP.get(label, "neutral")
    except Exception as e:
        print(f"  [XLM] error: {e}")
        return "neutral"

print("Running XLM-RoBERTa sentiment analysis...")
df_usable["sentiment_xlm"] = df_usable["analysis_text"].progress_apply(xlm_label)
print(df_usable["sentiment_xlm"].value_counts().to_string())

# =======================================================================
# STEP 5: DEEPSEEK ANALYSIS FUNCTION
# KEY FIX: YouTube records pass video_title as separate context so the
# model considers it ONCE as background, not as part of the opinion text
# =======================================================================
MAX_TEXT_CHARS = 1500

def safe_parse_json(raw: str):
    if not raw: return None
    cleaned = re.sub(r"^```(?:json)?\s*", "", raw.strip(), flags=re.IGNORECASE)
    cleaned = re.sub(r"\s*```$", "", cleaned.strip()).strip()
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError as e:
        print(f"    [JSON error] {e} | snippet: {raw[:80]}")
        return None

def analyze_with_deepseek(brand: str, text: str, video_title: str = "", max_tries: int = 3):
    """
    Analyse customer opinion text.
    video_title: if provided (YouTube), it's injected as background context ONCE,
                 separate from the opinion text so it doesn't skew every comment.
    """
    if not text or len(text.strip()) < 15:
        return None
    text = text.strip()[:MAX_TEXT_CHARS]

    # Build context block for YouTube
    if video_title and video_title.strip():
        context_block = (
            f"Video context (for background only — do NOT let it dominate the sentiment): "
            f"\"{video_title.strip()[:200]}\"\n\n"
        )
    else:
        context_block = ""

    prompt = f"""You are a brand analyst for {brand}.
{context_block}Analyze the following customer opinion and return ONLY valid JSON (no extra text, no markdown).

Return this exact structure:
{{
  "overall_sentiment": "positive|negative|neutral",
  "aspect_analysis": [
    {{"aspect": "...", "sentiment": "positive|negative|neutral", "reason": "..."}}
  ],
  "actionable_suggestions": ["...", "..."]
}}

Rules:
- Judge sentiment based on the customer opinion text below, NOT the video title above.
- The video title is only context; the comment/text is what matters for sentiment.
- Identify ALL relevant aspects (Food Quality, Staff Attitude, Wait Time, Price,
  Cleanliness, App Experience, Delivery, Packaging, Promotions, or any other).
- Give specific reasons citing the text. Give 2-3 actionable suggestions.

Customer opinion: \"{text}\"
"""
    headers  = {"Authorization": f"Bearer {DEEPSEEK_API_KEY}", "Content-Type": "application/json"}
    payload  = {"model": "deepseek-chat",
                "messages": [{"role": "user", "content": prompt}],
                "response_format": {"type": "json_object"},
                "temperature": 0.2, "max_tokens": 700}

    for attempt in range(max_tries):
        try:
            resp = requests.post(DEEPSEEK_URL, headers=headers, json=payload, timeout=60)
            if resp.status_code == 200:
                parsed = safe_parse_json(resp.json()["choices"][0]["message"]["content"])
                if parsed and "overall_sentiment" in parsed and "aspect_analysis" in parsed:
                    return parsed
                time.sleep(2)
            elif resp.status_code == 400:
                text = text[:600]
                payload["messages"][0]["content"] = prompt[:2000]
                time.sleep(2)
            elif resp.status_code == 429:
                wait = 15 + attempt * 15
                print(f"    [DeepSeek] Rate limited. Waiting {wait}s...")
                time.sleep(wait)
            else:
                print(f"    [DeepSeek] HTTP {resp.status_code}: {resp.text[:150]}")
                time.sleep(3)
        except requests.exceptions.Timeout:
            print(f"    [DeepSeek] Timeout attempt {attempt+1}")
            time.sleep(5)
        except Exception as e:
            print(f"    [DeepSeek] Error attempt {attempt+1}: {e}")
            time.sleep(3)
    return None

# =======================================================================
# STEP 6: BUILD SAMPLE
# (uses XLM sentiment to prioritise opinionated records)
# =======================================================================
SAMPLE_SIZE = 500

opinion_rows = df_usable[
    df_usable["sentiment_xlm"].isin(["positive", "negative"]) &
    (df_usable["analysis_text"].str.len() > 30)
]
neutral_rows = df_usable[
    (df_usable["sentiment_xlm"] == "neutral") &
    (df_usable["analysis_text"].str.len() > 30)
]

print(f"\nOpinionated: {len(opinion_rows)} | Neutral: {len(neutral_rows)}")

if len(opinion_rows) >= SAMPLE_SIZE:
    df_sample = opinion_rows.sample(SAMPLE_SIZE, random_state=42).copy()
else:
    df_sample = opinion_rows.copy()
    remaining = SAMPLE_SIZE - len(df_sample)
    if remaining > 0 and len(neutral_rows) > 0:
        df_sample = pd.concat([df_sample,
                               neutral_rows.sample(min(remaining, len(neutral_rows)), random_state=42)])

if df_sample.empty:
    df_sample = df_usable.copy()

df_sample = df_sample.drop_duplicates(subset=["_id"]).reset_index(drop=True)
print(f"Analysing {len(df_sample)} rows with DeepSeek...")

# =======================================================================
# STEP 7: RUN AI ANALYSIS
# For YouTube rows: pass video_title as separate context to fix title bias
# =======================================================================
def run_analysis(row):
    platform    = str(row.get("platform", "")).lower()
    video_title = clean_text(row.get("source_title", "")) if platform == "youtube" else ""
    return analyze_with_deepseek(BRAND, row["analysis_text"], video_title=video_title)

df_sample["ai_analysis"] = df_sample.progress_apply(run_analysis, axis=1)

success = df_sample["ai_analysis"].notna().sum()
print(f"\nDone: {success}/{len(df_sample)} succeeded.")

if success == 0:
    raise ValueError("All DeepSeek calls failed. Check API key and network.")

# =======================================================================
# STEP 8: AGGREGATE RESULTS
# =======================================================================
all_aspects, all_suggestions = [], []

for _, row in df_sample.dropna(subset=["ai_analysis"]).iterrows():
    analysis = row["ai_analysis"]
    if not isinstance(analysis, dict): continue
    platform = row.get("platform", "unknown")

    for a in (analysis.get("aspect_analysis") or []):
        if not isinstance(a, dict): continue
        all_aspects.append({
            "platform": platform,
            "aspect": str(a.get("aspect", "Other")).strip(),
            "sentiment": str(a.get("sentiment", "neutral")).lower().strip(),
            "reason": str(a.get("reason", "")),
            "record_id": row["_id"],
            "record_type": row.get("record_type", ""),
            "url": row.get("url", ""),
            "overall_sentiment": analysis.get("overall_sentiment", ""),
            "brand": BRAND,
        })
    for s in (analysis.get("actionable_suggestions") or []):
        if isinstance(s, str) and s.strip():
            all_suggestions.append({
                "platform": platform,
                "suggestion": s.strip(),
                "record_id": row["_id"],
                "brand": BRAND,
            })

df_aspects     = pd.DataFrame(all_aspects)     if all_aspects     else pd.DataFrame(columns=["platform","aspect","sentiment","reason","record_id","record_type","url","overall_sentiment","brand"])
df_suggestions = pd.DataFrame(all_suggestions) if all_suggestions else pd.DataFrame(columns=["platform","suggestion","record_id","brand"])

# Also update records with XLM sentiment
ai_sent_map = {}
for _, row in df_sample.dropna(subset=["ai_analysis"]).iterrows():
    ai_sent_map[row["_id"]] = row["ai_analysis"].get("overall_sentiment", "")

df_sample["ai_overall_sentiment"] = df_sample["_id"].map(ai_sent_map).fillna("")

print(f"Aspects: {len(df_aspects)} | Suggestions: {len(df_suggestions)}")

# =======================================================================
# STEP 9: SUMMARIZE SUGGESTIONS
# =======================================================================
def summarize_suggestions_with_ai(suggestion_list):
    uniq = list(dict.fromkeys([s.strip() for s in suggestion_list if s.strip()]))[:150]
    if not uniq: return "No suggestions to summarize."
    print(f"\nSummarising {len(uniq)} suggestions...")
    prompt = f"""You are an executive strategist for {BRAND}.
Synthesize these {len(uniq)} raw suggestions into exactly 5 high-level strategic recommendations.
Group similar ideas. Be concise, professional and specific.

RAW SUGGESTIONS:
{chr(10).join(uniq)}

Return a numbered list of exactly 5 strategic recommendations."""
    headers = {"Authorization": f"Bearer {DEEPSEEK_API_KEY}", "Content-Type": "application/json"}
    payload = {"model": "deepseek-chat",
               "messages": [{"role": "user", "content": prompt[:4000]}],
               "max_tokens": 600}
    try:
        r = requests.post(DEEPSEEK_URL, headers=headers, json=payload, timeout=90)
        if r.status_code == 200:
            return r.json()["choices"][0]["message"]["content"]
        return f"Summary error: HTTP {r.status_code}"
    except Exception as e:
        return f"Summary error: {e}"

final_summary = summarize_suggestions_with_ai(df_suggestions["suggestion"].tolist()) if not df_suggestions.empty else "No suggestions generated."

# =======================================================================
# STEP 10: DISPLAY INSIGHTS
# =======================================================================
print("\n" + "="*60)
print(f"  BRANDPULSE INSIGHTS — {BRAND}")
print("="*60)

if not df_aspects.empty:
    print("\n--- Top Aspects ---")
    print(df_aspects["aspect"].value_counts().head(15).to_string())
    print("\n--- Sentiment per Aspect ---")
    pivot = df_aspects.groupby("aspect")["sentiment"].value_counts().unstack(fill_value=0)
    pivot["total"] = pivot.sum(axis=1)
    print(pivot.sort_values("total", ascending=False).drop(columns="total").to_string())
    print("\n--- Overall Sentiment Distribution ---")
    print(df_aspects["overall_sentiment"].value_counts().to_string())

print("\n--- Strategic Recommendations ---")
print(final_summary)

print("\n--- df_aspects preview ---")
display(df_aspects.head(10))
print("\n--- df_suggestions preview ---")
display(df_suggestions.head(10))


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 15.0 MB/s eta 0:00:00
Loading secrets...
  DeepSeek key loaded from secret 'deepseekAPI'
  MongoDB URI loaded from secret 'MONGO_URI'

  Brand name [KFC]: KFC

Ready. Brand='KFC' | Key starts with: 'sk-31762...'

Connecting to MongoDB...
Loaded 1503 records for brand 'KFC'
platform     record_type
google_maps  review          193
reddit       comment          70
             post             30
youtube      comment        1171
             video            39

Usable rows: 1393 / 1503

Loading XLM-RoBERTa multilingual sentiment model...
(First run downloads ~1.1 GB — subsequent runs use cache)


config.json:   0%|          | 0.00/841 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

...

  0%|          | 0/500 [00:00<?, ?it/s]


Done: 500/500 succeeded.
Aspects: 827 | Suggestions: 1496

Summarising 150 suggestions...

  BRANDPULSE INSIGHTS — KFC

--- Top Aspects ---
aspect
Food Quality          304
Other                 110
Staff Attitude        104
Price                  51
Wait Time              41
Promotions             36
Cleanliness            27
Service                14
Overall Experience     11
Customer Service       11
Delivery               10
App Experience          7
Location                6
Packaging               5
Portion Size            5

--- Sentiment per Aspect ---
sentiment                 negative  neutral  positive
aspect                                               
Food Quality                   155       32       117
Other                           21       43        46
Staff Attitude                  60        7        37
Price                           39        6         6
Wait Time                       32        5         4
Promotions                      18        8        10


,platform,aspect,sentiment,reason,record_id,record_type,url,overall_sentiment,brand
0,youtube,Food Quality,negative,Customer says 'they sell old chicken' and 'all...,69b2dd892ecfa2b4ec96b878,comment,https://www.youtube.com/watch?v=6lNCokr-1Zg,negative,KFC
1,google_maps,Wait Time,negative,Waited for more than 40 minutes to take the order,69b2ded12ecfa2b4ec96ba99,review,https://serpapi.com/search.json?data_id=0x3ae2...,negative,KFC
2,google_maps,Service,negative,Very poor services,69b2ded12ecfa2b4ec96ba99,review,https://serpapi.com/search.json?data_id=0x3ae2...,negative,KFC
3,reddit,Food Quality,negative,The customer implies KFC is unhealthy and not ...,69b287a92ecfa2b4ec96b4d6,comment,https://www.reddit.com/r/srilanka/comments/1o5...,negative,KFC
4,reddit,Price,negative,"The customer mentions cheaper options, indicat...",69b287a92ecfa2b4ec96b4d6,comment,https://www.reddit.com/r/srilanka/comments/1o5...,negative,KFC
5,youtube,Food Quality,positive,"The customer recommends Malaysian KFC, implyin...",69b2dda12ecfa2b4ec96b8e0,comment,https://www.youtube.com/watch?v=A9TvJm74YHU,positive,KFC
6,youtube,Food Quality,negative,Customer blames KFC for food poisoning that ma...,69b2da502ecfa2b4ec96b55f,video,https://www.youtube.com/watch?v=GUImROmrdok,negative,KFC
7,youtube,Cleanliness,negative,Customer mentions 'sanitation purposes mainly'...,69b2da502ecfa2b4ec96b55f,video,https://www.youtube.com/watch?v=GUImROmrdok,negative,KFC
8,youtube,Staff Attitude,positive,The drive-thru lady complimented the customer'...,69b2da502ecfa2b4ec96b55f,video,https://www.youtube.com/watch?v=GUImROmrdok,negative,KFC
9,youtube,Wait Time,neutral,No explicit complaint about wait time; the cus...,69b2da502ecfa2b4ec96b55f,video,https://www.youtube.com/watch?v=GUImROmrdok,negative,KFC



--- df_suggestions preview ---


,platform,suggestion,record_id,brand
0,youtube,Implement strict freshness checks and rotation...,69b2dd892ecfa2b4ec96b878,KFC
1,youtube,Conduct mystery audits to ensure food quality ...,69b2dd892ecfa2b4ec96b878,KFC
2,youtube,Launch a customer feedback system to quickly a...,69b2dd892ecfa2b4ec96b878,KFC
3,google_maps,Improve order taking efficiency to reduce wait...,69b2ded12ecfa2b4ec96ba99,KFC
4,google_maps,Implement a queue management system to monitor...,69b2ded12ecfa2b4ec96ba99,KFC
5,google_maps,Train staff to handle peak hours more effectiv...,69b2ded12ecfa2b4ec96ba99,KFC
6,reddit,Highlight healthier menu options or nutritiona...,69b287a92ecfa2b4ec96b4d6,KFC
7,reddit,Introduce value meals or promotions to address...,69b287a92ecfa2b4ec96b4d6,KFC
8,reddit,Emphasize unique taste or experience that just...,69b287a92ecfa2b4ec96b4d6,KFC
9,youtube,Consider promoting Malaysian KFC menu items in...,69b2dda12ecfa2b4ec96b8e0,KFC


## Cell 6 — Save Analysis Results to MongoDB

> **Must run in the same session as Cell 5.**

Writes to:
- `analysis_aspects` — every aspect row (what the dashboard reads)
- `analysis_suggestions` — individual suggestions
- `analysis_summary` — strategic summary text
- `records` — back-populated with `ai_overall_sentiment`

In [ ]:
# =======================================================================
# BrandPulse — Save DeepSeek Analysis to MongoDB
# Writes to collections that the Streamlit dashboard reads:
#   analysis_aspects     ← df_aspects rows
#   analysis_suggestions ← df_suggestions rows
#   analysis_summary     ← final_summary text (one doc)
#   records              ← updated with ai_overall_sentiment etc.
# =======================================================================

!pip -q install pymongo certifi

import pandas as pd
import certifi
from pymongo import MongoClient, ASCENDING, UpdateOne
from datetime import datetime, timezone

# ── Load secrets (reuse from session if already loaded) ──────────────────────
try:
    from google.colab import userdata
    _MONGO_URI     = ""
    _MONGO_DB_NAME = "brandpulse"
    _BRAND         = "KFC"
    for name in ["MONGO_URI", "mongo_uri"]:
        try:
            v = (userdata.get(name) or "").strip()
            if v: _MONGO_URI = v; break
        except: pass
    for name in ["MONGO_DB_NAME", "mongo_db"]:
        try:
            v = (userdata.get(name) or "").strip()
            if v: _MONGO_DB_NAME = v; break
        except: pass
    for name in ["BRAND", "brand"]:
        try:
            v = (userdata.get(name) or "").strip()
            if v: _BRAND = v; break
        except: pass
except:
    import getpass
    _MONGO_URI     = getpass.getpass("MongoDB URI: ")
    _MONGO_DB_NAME = input("DB name [brandpulse]: ").strip() or "brandpulse"
    _BRAND         = input("Brand [KFC]: ").strip() or "KFC"

# If the analysis notebook already set these globals, prefer those
MONGO_URI     = globals().get("MONGO_URI",     _MONGO_URI)
MONGO_DB_NAME = globals().get("MONGO_DB_NAME", _MONGO_DB_NAME)
BRAND         = globals().get("BRAND",         _BRAND)

print(f"Brand: {BRAND} | DB: {MONGO_DB_NAME}")

# ── Connect ───────────────────────────────────────────────────────────────────
_cli = MongoClient(MONGO_URI, tls=True, tlsCAFile=certifi.where(), serverSelectionTimeoutMS=30000)
_cli.admin.command("ping")
_db = _cli[MONGO_DB_NAME]
print("Connected to MongoDB ✅")

now_utc = datetime.now(timezone.utc)

# ── Validate variables from analysis session ──────────────────────────────────
def _get_df(name):
    v = globals().get(name)
    if v is None:
        print(f"  WARNING: '{name}' not in session — run Analysis notebook first.")
        return None
    if not isinstance(v, pd.DataFrame) or v.empty:
        print(f"  WARNING: '{name}' is empty.")
        return None
    print(f"  '{name}': {len(v)} rows ✅")
    return v

print("\nChecking session variables...")
_aspects_df  = _get_df("df_aspects")
_sugg_df     = _get_df("df_suggestions")
_sample_df   = _get_df("df_sample")
_summary_txt = globals().get("final_summary") or globals().get("final_suggestion_summary") or ""

if _aspects_df is None and _sugg_df is None:
    raise ValueError(
        "No analysis data found in session.\n"
        "Run the Analysis notebook (BrandPulse_Analysis_V3) FIRST in the same session."
    )

# ── SAVE 1: analysis_aspects ─────────────────────────────────────────────────
if _aspects_df is not None:
    print("\n--- Saving → analysis_aspects ---")
    coll = _db["analysis_aspects"]
    coll.create_index(
        [("brand", ASCENDING), ("record_id", ASCENDING), ("aspect", ASCENDING)],
        unique=True, sparse=True, name="uniq_brand_record_aspect"
    )
    ops = []
    for _, row in _aspects_df.iterrows():
        doc = {
            "brand":             BRAND,
            "platform":          str(row.get("platform",         "") or ""),
            "aspect":            str(row.get("aspect",           "") or "Other"),
            "sentiment":         str(row.get("sentiment",        "") or "neutral").lower().strip(),
            "reason":            str(row.get("reason",           "") or ""),
            "record_id":         str(row.get("record_id",        "") or ""),
            "record_type":       str(row.get("record_type",      "") or ""),
            "url":               str(row.get("url",              "") or ""),
            "overall_sentiment": str(row.get("overall_sentiment","") or ""),
            "saved_at":          now_utc,
        }
        flt = {"brand": doc["brand"], "record_id": doc["record_id"], "aspect": doc["aspect"]}
        ops.append(UpdateOne(flt, {"$set": doc}, upsert=True))
    if ops:
        r = _db["analysis_aspects"].bulk_write(ops, ordered=False)
        print(f"  Saved: {r.upserted_count} new | {r.modified_count} updated")

# ── SAVE 2: analysis_suggestions ─────────────────────────────────────────────
if _sugg_df is not None:
    print("\n--- Saving → analysis_suggestions ---")
    coll = _db["analysis_suggestions"]
    coll.create_index(
        [("brand", ASCENDING), ("record_id", ASCENDING), ("suggestion", ASCENDING)],
        unique=True, sparse=True, name="uniq_brand_record_sugg"
    )
    ops = []
    for _, row in _sugg_df.iterrows():
        s = str(row.get("suggestion", "") or "").strip()
        if not s:
            continue
        doc = {
            "brand":      BRAND,
            "platform":   str(row.get("platform",  "") or ""),
            "suggestion": s,
            "record_id":  str(row.get("record_id", "") or ""),
            "saved_at":   now_utc,
        }
        flt = {"brand": doc["brand"], "record_id": doc["record_id"], "suggestion": doc["suggestion"]}
        ops.append(UpdateOne(flt, {"$set": doc}, upsert=True))
    if ops:
        r = _db["analysis_suggestions"].bulk_write(ops, ordered=False)
        print(f"  Saved: {r.upserted_count} new | {r.modified_count} updated")

# ── SAVE 3: analysis_summary (strategic recommendations text) ─────────────────
print("\n--- Saving → analysis_summary ---")
if _summary_txt and isinstance(_summary_txt, str) and len(_summary_txt.strip()) > 10:
    _db["analysis_summary"].update_one(
        {"brand": BRAND},
        {"$set": {
            "brand":   BRAND,
            "summary": _summary_txt.strip(),
            "saved_at": now_utc,
        }},
        upsert=True
    )
    print(f"  Summary saved ({len(_summary_txt)} chars) ✅")
else:
    print("  No summary text found in session variable 'final_summary'.")
    print("  Run the full analysis notebook to generate it.")

# ── SAVE 4: Back-write AI results into records collection ─────────────────────
if _sample_df is not None:
    print("\n--- Updating records with AI results ---")
    from bson import ObjectId
    ops = []
    for _, row in _sample_df.dropna(subset=["ai_analysis"]).iterrows():
        a = row.get("ai_analysis")
        if not isinstance(a, dict):
            continue
        upd = {
            "ai_overall_sentiment": a.get("overall_sentiment", ""),
            "ai_aspect_analysis":   a.get("aspect_analysis", []),
            "ai_suggestions":       a.get("actionable_suggestions", []),
            "ai_aspect_count":      len(a.get("aspect_analysis", [])),
            "ai_analysed":          True,
            "ai_analysed_at":       now_utc,
        }
        try:
            flt = {"_id": ObjectId(str(row["_id"]))}
        except Exception:
            flt = {"_id": str(row["_id"])}
        ops.append(UpdateOne(flt, {"$set": upd}))
    if ops:
        r = _db["records"].bulk_write(ops, ordered=False)
        print(f"  Records updated: {r.modified_count}/{len(ops)} ✅")

# ── Verification ──────────────────────────────────────────────────────────────
print("\n" + "="*55)
print("  VERIFICATION — MongoDB Collections")
print("="*55)
print(f"  analysis_aspects    : {_db['analysis_aspects'].count_documents({'brand': BRAND})} rows")
print(f"  analysis_suggestions: {_db['analysis_suggestions'].count_documents({'brand': BRAND})} rows")
summary_exists = _db['analysis_summary'].count_documents({'brand': BRAND}) > 0
print(f"  analysis_summary    : {'✅ exists' if summary_exists else '❌ missing'}")
print(f"  records (ai-tagged) : {_db['records'].count_documents({'brand': BRAND, 'ai_analysed': True})} records")
print(f"\n  All collections: {_db.list_collection_names()}")
print("\n✅ All analysis results saved. Dashboard will now show data.")


Brand: KFC | DB: brandpulse
Connected to MongoDB ✅

Checking session variables...
  'df_aspects': 827 rows ✅
  'df_suggestions': 1496 rows ✅
  'df_sample': 500 rows ✅

--- Saving → analysis_aspects ---
  Saved: 771 new | 56 updated

--- Saving → analysis_suggestions ---
  Saved: 1496 new | 0 updated

--- Saving → analysis_summary ---
  Summary saved (3259 chars) ✅

--- Updating records with AI results ---
  Records updated: 500/500 ✅

  VERIFICATION — MongoDB Collections
  analysis_aspects    : 999 rows
  analysis_suggestions: 1791 rows
  analysis_summary    : ✅ exists
  records (ai-tagged) : 500 records

  All collections: ['analysis_suggestions', 'fs.files', 'records', 'analysis_summary', 'fs.chunks', 'analysis_aspects']

✅ All analysis results saved. Dashboard will now show data.


## Cell 7 — QA & Validation

In [ ]:
# =======================================================================
# BrandPulse — QA & Validation Notebook
# Run this after scraping + analysis to verify everything worked correctly
# =======================================================================

!pip -q install pymongo pandas certifi tabulate

import pandas as pd
import json
import re
import os
import certifi
from pymongo import MongoClient
from datetime import datetime, timezone, timedelta
from IPython.display import display, HTML
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.max_rows", 50)

# =======================================================================
# LOAD SECRETS (same as analysis notebook — no re-entering)
# =======================================================================
try:
    from google.colab import userdata
    MONGO_URI     = ""
    MONGO_DB_NAME = "brandpulse"
    BRAND         = "KFC"
    for name in ["MONGO_URI", "mongo_uri"]:
        try:
            v = (userdata.get(name) or "").strip()
            if v: MONGO_URI = v; break
        except: pass
    for name in ["MONGO_DB_NAME", "mongo_db"]:
        try:
            v = (userdata.get(name) or "").strip()
            if v: MONGO_DB_NAME = v; break
        except: pass
    for name in ["BRAND", "brand"]:
        try:
            v = (userdata.get(name) or "").strip()
            if v: BRAND = v; break
        except: pass
except:
    import getpass
    MONGO_URI     = getpass.getpass("MongoDB URI: ")
    MONGO_DB_NAME = input("DB name [brandpulse]: ").strip() or "brandpulse"
    BRAND         = input("Brand [KFC]: ").strip() or "KFC"

client = MongoClient(MONGO_URI, tls=True, tlsCAFile=certifi.where(), serverSelectionTimeoutMS=30000)
client.admin.command("ping")
db   = client[MONGO_DB_NAME]
coll = db["records"]

print(f"Connected to MongoDB | DB: {MONGO_DB_NAME} | Brand: {BRAND}")
print("="*60)

# Helper to print pass/fail
def qa_check(name, condition, detail=""):
    status = "✅ PASS" if condition else "❌ FAIL"
    print(f"  {status}  {name}")
    if detail:
        print(f"          → {detail}")
    return condition

results = []  # collect all pass/fail for summary

# =======================================================================
# QA 1 — SCRAPER: Was data actually collected?
# =======================================================================
print("\n" + "="*60)
print("QA 1: SCRAPER — Data Collection Checks")
print("="*60)

total = coll.count_documents({"brand": BRAND})
r1 = qa_check("Total records > 0", total > 0, f"{total} records found for brand '{BRAND}'")
results.append(("Total records > 0", r1))

# Per-platform counts
platforms = ["twitter", "reddit", "youtube", "google_maps"]
platform_counts = {}
for p in platforms:
    count = coll.count_documents({"brand": BRAND, "platform": p})
    platform_counts[p] = count
    r = qa_check(f"  {p} has records", count > 0, f"{count} records")
    results.append((f"{p} has records", r))

print("\n  Platform breakdown:")
df_plat = pd.DataFrame([
    {"platform": p, "count": c} for p, c in platform_counts.items()
])
display(df_plat)

# Record type breakdown
print("\n  Record type breakdown:")
pipeline = [
    {"$match": {"brand": BRAND}},
    {"$group": {"_id": {"platform": "$platform", "record_type": "$record_type"}, "count": {"$sum": 1}}},
    {"$sort": {"_id.platform": 1, "_id.record_type": 1}}
]
breakdown = list(coll.aggregate(pipeline))
df_types = pd.DataFrame([{
    "platform": b["_id"]["platform"],
    "record_type": b["_id"]["record_type"],
    "count": b["count"]
} for b in breakdown])
if not df_types.empty:
    display(df_types)

# Date range check
print("\n  Date range of collected data:")
newest = coll.find_one({"brand": BRAND}, sort=[("created_at", -1)])
oldest = coll.find_one({"brand": BRAND}, sort=[("created_at", 1)])
if newest and oldest:
    print(f"    Oldest: {oldest.get('created_at', 'N/A')}")
    print(f"    Newest: {newest.get('created_at', 'N/A')}")

# Check records have text
with_text = coll.count_documents({
    "brand": BRAND,
    "text": {"$exists": True, "$nin": ["", None]}
})
r2 = qa_check("Records with non-empty text", with_text > 0,
              f"{with_text}/{total} records have text ({100*with_text//total if total else 0}%)")
results.append(("Records have text", r2))

# YouTube captions
yt_total   = coll.count_documents({"brand": BRAND, "platform": "youtube", "record_type": "video"})
yt_caps    = coll.count_documents({"brand": BRAND, "platform": "youtube",
                                    "captions_text": {"$nin": ["", None]}})
yt_trans   = coll.count_documents({"brand": BRAND, "platform": "youtube",
                                    "transcript_text": {"$nin": ["", None]}})
if yt_total > 0:
    r3 = qa_check(f"YouTube: videos with captions or transcript",
                  (yt_caps + yt_trans) > 0,
                  f"{yt_caps} have captions, {yt_trans} have transcript out of {yt_total} videos")
    results.append(("YouTube captions/transcripts", r3))

# Google reviews have ratings
gr_total  = coll.count_documents({"brand": BRAND, "platform": "google_maps"})
gr_rated  = coll.count_documents({"brand": BRAND, "platform": "google_maps",
                                   "rating": {"$exists": True, "$nin": ["", None]}})
if gr_total > 0:
    r4 = qa_check("Google reviews have ratings", gr_rated > 0,
                  f"{gr_rated}/{gr_total} reviews have star ratings")
    results.append(("Google reviews have ratings", r4))

# Duplicate check
print("\n  Checking for duplicate records...")
dup_pipeline = [
    {"$match": {"brand": BRAND}},
    {"$group": {
        "_id": {"platform": "$platform", "record_type": "$record_type", "source_id": "$source_id"},
        "count": {"$sum": 1}
    }},
    {"$match": {"count": {"$gt": 1}}},
    {"$count": "duplicates"}
]
dup_result = list(coll.aggregate(dup_pipeline))
dup_count  = dup_result[0]["duplicates"] if dup_result else 0
r5 = qa_check("No duplicate records", dup_count == 0, f"{dup_count} duplicates found")
results.append(("No duplicates", r5))

# =======================================================================
# QA 2 — TEXT QUALITY: Is the stored text useful?
# =======================================================================
print("\n" + "="*60)
print("QA 2: TEXT QUALITY — Content Checks")
print("="*60)

# Load sample into DataFrame
docs = list(coll.find(
    {"brand": BRAND},
    {"platform": 1, "record_type": 1, "text": 1, "source_title": 1,
     "captions_text": 1, "transcript_text": 1, "lang": 1, "url": 1}
))
df = pd.DataFrame(docs)
for col in ["text", "source_title", "captions_text", "transcript_text", "lang", "url"]:
    if col not in df.columns: df[col] = ""
    df[col] = df[col].fillna("")

df["combined_text"] = (
    df["source_title"] + " " + df["text"] + " " +
    df["captions_text"] + " " + df["transcript_text"]
).str.strip()
df["text_len"] = df["combined_text"].str.len()

# Text length distribution
print("\n  Text length distribution (characters):")
print(df.groupby("platform")["text_len"].describe()[["count","min","mean","max"]].to_string())

# Short text warning
very_short = (df["text_len"] < 15).sum()
r6 = qa_check("Less than 20% records are very short (<15 chars)",
              very_short / len(df) < 0.2 if len(df) > 0 else False,
              f"{very_short}/{len(df)} records have <15 chars of text")
results.append(("Text length OK", r6))

# Language check (basic)
def has_sinhala(text):
    return any("\u0D80" <= ch <= "\u0DFF" for ch in str(text))

df["is_sinhala"] = df["combined_text"].apply(has_sinhala)
sinhala_count    = df["is_sinhala"].sum()
print(f"\n  Records with Sinhala text: {sinhala_count} ({100*sinhala_count//len(df) if len(df) else 0}%)")

# Show sample texts per platform
print("\n  Sample texts per platform:")
for p in platforms:
    sample = df[df["platform"] == p]["combined_text"].dropna()
    sample = sample[sample.str.len() > 20]
    if not sample.empty:
        print(f"\n  [{p.upper()}] Sample:")
        for t in sample.head(2).values:
            print(f"    → {t[:120]}")

# URL format check
urls_present = (df["url"].str.startswith("http")).sum()
r7 = qa_check("Records have valid URLs",
              urls_present > total * 0.5,
              f"{urls_present}/{total} records have http URLs")
results.append(("URLs valid", r7))

# =======================================================================
# QA 3 — DEEPSEEK ANALYSIS: Did the AI analysis produce sensible output?
# =======================================================================
print("\n" + "="*60)
print("QA 3: DEEPSEEK ANALYSIS — Output Validation")
print("="*60)

print("""
NOTE: This section checks whether df_aspects and df_suggestions exist
from the analysis notebook. Run the analysis notebook first, then re-run
this QA cell to validate the AI output.
""")

try:
    # Check df_aspects exists and is not empty
    r8 = qa_check("df_aspects exists and has rows",
                  "df_aspects" in dir() and not df_aspects.empty,
                  f"{len(df_aspects) if 'df_aspects' in dir() else 0} aspect rows")
    results.append(("df_aspects not empty", r8))

    if "df_aspects" in dir() and not df_aspects.empty:
        # Sentiment values are valid
        valid_sentiments = {"positive", "negative", "neutral"}
        bad_sentiments   = ~df_aspects["sentiment"].isin(valid_sentiments)
        r9 = qa_check("All sentiment values are valid (pos/neg/neutral)",
                      bad_sentiments.sum() == 0,
                      f"{bad_sentiments.sum()} rows have unexpected sentiment values")
        results.append(("Valid sentiments", r9))

        # Aspect names are not empty
        empty_aspects = (df_aspects["aspect"].str.strip() == "").sum()
        r10 = qa_check("No empty aspect names", empty_aspects == 0,
                       f"{empty_aspects} rows have blank aspect field")
        results.append(("No empty aspects", r10))

        # Reason field is populated
        empty_reasons = (df_aspects["reason"].str.strip() == "").sum()
        r11 = qa_check("Less than 30% aspects have empty reasons",
                       empty_reasons / len(df_aspects) < 0.3,
                       f"{empty_reasons}/{len(df_aspects)} aspects have empty reason")
        results.append(("Reasons populated", r11))

        # Aspect variety check
        unique_aspects = df_aspects["aspect"].nunique()
        r12 = qa_check("At least 3 distinct aspects identified",
                       unique_aspects >= 3,
                       f"{unique_aspects} unique aspects found: {df_aspects['aspect'].unique().tolist()}")
        results.append(("Aspect variety", r12))

        # Sentiment balance check (not 100% positive or 100% negative)
        sent_counts = df_aspects["sentiment"].value_counts(normalize=True)
        dominant    = sent_counts.max()
        r13 = qa_check("Sentiment not 100% one-sided (>95% same)",
                       dominant < 0.95,
                       f"Sentiment distribution: {sent_counts.round(2).to_dict()}")
        results.append(("Sentiment balanced", r13))

        print("\n  Aspect distribution:")
        display(df_aspects["aspect"].value_counts().head(15))

        print("\n  Sentiment per aspect:")
        pivot = df_aspects.groupby("aspect")["sentiment"].value_counts().unstack(fill_value=0)
        display(pivot)

        print("\n  Sample AI reasons (negative):")
        neg_sample = df_aspects[df_aspects["sentiment"] == "negative"][["aspect","reason"]].head(5)
        display(neg_sample)

except Exception as e:
    print(f"  Could not run analysis QA: {e}")
    print("  → Run the analysis notebook first, then re-run this cell.")

try:
    r14 = qa_check("df_suggestions exists and has rows",
                   "df_suggestions" in dir() and not df_suggestions.empty,
                   f"{len(df_suggestions) if 'df_suggestions' in dir() else 0} suggestions")
    results.append(("df_suggestions not empty", r14))

    if "df_suggestions" in dir() and not df_suggestions.empty:
        vague = df_suggestions["suggestion"].str.len() < 20
        r15 = qa_check("Suggestions are specific (>20 chars)",
                       vague.sum() / len(df_suggestions) < 0.2,
                       f"{vague.sum()} out of {len(df_suggestions)} are very short/vague")
        results.append(("Suggestions specific", r15))
except Exception as e:
    print(f"  Could not check df_suggestions: {e}")

# =======================================================================
# QA 4 — EDGE CASES: Known failure modes
# =======================================================================
print("\n" + "="*60)
print("QA 4: EDGE CASE CHECKS")
print("="*60)

# Check: records with brand name actually mentioned in text
brand_mentioned = coll.count_documents({
    "brand": BRAND,
    "text": {"$regex": BRAND, "$options": "i"}
})
r16 = qa_check(f"Brand name '{BRAND}' mentioned in text records",
               brand_mentioned > 0,
               f"{brand_mentioned} records explicitly mention '{BRAND}' in text")
results.append(("Brand mentioned in text", r16))

# Check: no records with future dates
try:
    future_count = sum(
        1 for doc in coll.find({"brand": BRAND, "created_at": {"$regex": "^20"}}, {"created_at": 1})
        if doc.get("created_at", "") > datetime.now().strftime("%Y-%m-%d")
    )
    r17 = qa_check("No records with future dates", future_count == 0,
                   f"{future_count} records have future created_at")
    results.append(("No future dates", r17))
except:
    pass

# Check: Google reviews have addresses
if gr_total > 0:
    gr_with_addr = coll.count_documents({
        "brand": BRAND, "platform": "google_maps",
        "location_address": {"$nin": ["", None]}
    })
    r18 = qa_check("Google reviews have location addresses",
                   gr_with_addr > 0,
                   f"{gr_with_addr}/{gr_total} reviews have addresses")
    results.append(("Google reviews have addresses", r18))

# Check: YouTube videos have titles
yt_with_title = coll.count_documents({
    "brand": BRAND, "platform": "youtube",
    "source_title": {"$nin": ["", None]}
})
if yt_total > 0:
    r19 = qa_check("YouTube records have titles",
                   yt_with_title == yt_total,
                   f"{yt_with_title}/{yt_total} YouTube records have titles")
    results.append(("YouTube titles present", r19))

# =======================================================================
# FINAL SUMMARY
# =======================================================================
print("\n" + "="*60)
print("QA SUMMARY")
print("="*60)

passed = sum(1 for _, r in results if r)
failed = sum(1 for _, r in results if not r)
total_checks = len(results)

print(f"\n  Total checks : {total_checks}")
print(f"  ✅ Passed    : {passed}")
print(f"  ❌ Failed    : {failed}")
print(f"  Score        : {100*passed//total_checks if total_checks else 0}%\n")

for name, result in results:
    icon = "✅" if result else "❌"
    print(f"  {icon}  {name}")

if failed == 0:
    print("\n🎉 ALL CHECKS PASSED — BrandPulse pipeline is working correctly!")
elif failed <= 2:
    print(f"\n⚠️  {failed} minor issue(s) found — review the ❌ items above.")
else:
    print(f"\n🚨 {failed} checks failed — review errors above before using results.")

print("\n" + "="*60)
print("QA COMPLETE")
print("="*60)


Connected to MongoDB | DB: brandpulse | Brand: KFC

QA 1: SCRAPER — Data Collection Checks
  ✅ PASS  Total records > 0
          → 1503 records found for brand 'KFC'
  ❌ FAIL    twitter has records
          → 0 records
  ✅ PASS    reddit has records
          → 100 records
  ✅ PASS    youtube has records
          → 1210 records
  ✅ PASS    google_maps has records
          → 193 records

  Platform breakdown:


,platform,count
0,twitter,0
1,reddit,100
2,youtube,1210
3,google_maps,193



  Record type breakdown:


,platform,record_type,count
0,google_maps,review,193
1,reddit,comment,70
2,reddit,post,30
3,youtube,comment,1171
4,youtube,video,39



  Date range of collected data:
    Oldest: 1 month ago
    Newest: 9 months ago
  ✅ PASS  Records with non-empty text
          → 1488/1503 records have text (99%)
  ✅ PASS  YouTube: videos with captions or transcript
          → 0 have captions, 37 have transcript out of 39 videos
  ✅ PASS  Google reviews have ratings
          → 193/193 reviews have star ratings

  Checking for duplicate records...
  ✅ PASS  No duplicate records
          → 0 duplicates found

QA 2: TEXT QUALITY — Content Checks

  Text length distribution (characters):
              count   min        mean      max
platform                                      
google_maps   193.0  25.0  247.222798   2141.0
reddit        100.0  29.0  379.560000   4926.0
youtube      1210.0  26.0  258.709917  18614.0
  ✅ PASS  Less than 20% records are very short (<15 chars)
          → 0/1503 records have <15 chars of text

  Records with Sinhala text: 560 (37%)

  Sample texts per platform:

  [REDDIT] Sample:
    → Any good burg

...


  Sentiment per aspect:


sentiment,negative,neutral,positive
aspect,,,
Accommodation,0,0,1
Ambiance,0,0,4
Ambience,0,0,1
Animal Welfare,1,0,0
App Experience,5,2,0
...,...,...,...
Video Content,0,0,2
Video Quality,0,0,1
Video Style,1,0,0



  Sample AI reasons (negative):


,aspect,reason
0,Food Quality,Customer says 'they sell old chicken' and 'all the Sri Lankan KFC is very bad'.
1,Wait Time,Waited for more than 40 minutes to take the order
2,Service,Very poor services
3,Food Quality,"The customer implies KFC is unhealthy and not worth eating, suggesting there..."
4,Price,"The customer mentions cheaper options, indicating KFC is perceived as overpr..."


  ✅ PASS  df_suggestions exists and has rows
          → 1496 suggestions
  ✅ PASS  Suggestions are specific (>20 chars)
          → 0 out of 1496 are very short/vague

QA 4: EDGE CASE CHECKS
  ✅ PASS  Brand name 'KFC' mentioned in text records
          → 448 records explicitly mention 'KFC' in text
  ❌ FAIL  No records with future dates
          → 5 records have future created_at
  ✅ PASS  Google reviews have location addresses
          → 193/193 reviews have addresses
  ❌ FAIL  YouTube records have titles
          → 1210/39 YouTube records have titles

QA SUMMARY

  Total checks : 23
  ✅ Passed    : 20
  ❌ Failed    : 3
  Score        : 86%

  ✅  Total records > 0
  ❌  twitter has records
  ✅  reddit has records
  ✅  youtube has records
  ✅  google_maps has records
  ✅  Records have text
  ✅  YouTube captions/transcripts
  ✅  Google reviews have ratings
  ✅  No duplicates
  ✅  Text length OK
  ✅  URLs valid
  ✅  df_aspects not empty
  ✅  Valid sentiments
  ✅  No empty aspects
  ✅

In [ ]:
# =======================================================================
# BrandPulse — DeepSeek Analysis Viewer
# Shows each analysed record side-by-side with DeepSeek's output
# so you can manually check accuracy
# =======================================================================

!pip -q install pymongo certifi pandas

import pandas as pd
import certifi
from pymongo import MongoClient
from IPython.display import display, HTML

# ── Load secrets ──────────────────────────────────────────────────────────────
try:
    from google.colab import userdata
    MONGO_URI     = ""
    MONGO_DB_NAME = "brandpulse"
    BRAND         = "KFC"
    for name in ["MONGO_URI", "mongo_uri"]:
        try:
            v = str(userdata.get(name) or "").strip()
            if v: MONGO_URI = v; break
        except: pass
    for name in ["MONGO_DB_NAME", "mongo_db"]:
        try:
            v = str(userdata.get(name) or "").strip()
            if v: MONGO_DB_NAME = v; break
        except: pass
    for name in ["BRAND", "brand"]:
        try:
            v = str(userdata.get(name) or "").strip()
            if v: BRAND = v; break
        except: pass
except:
    import getpass
    MONGO_URI     = getpass.getpass("MongoDB URI: ")
    MONGO_DB_NAME = input("DB name [brandpulse]: ").strip() or "brandpulse"
    BRAND         = input("Brand [KFC]: ").strip() or "KFC"

# Use session vars if already loaded
MONGO_URI     = globals().get("MONGO_URI",     MONGO_URI)
MONGO_DB_NAME = globals().get("MONGO_DB_NAME", MONGO_DB_NAME)
BRAND         = globals().get("BRAND",         BRAND)

# ── Connect ───────────────────────────────────────────────────────────────────
_cli = MongoClient(MONGO_URI, tls=True, tlsCAFile=certifi.where(), serverSelectionTimeoutMS=30000)
_db  = _cli[MONGO_DB_NAME]

# ── Load analysed records from MongoDB ───────────────────────────────────────
print(f"Loading analysed records for brand='{BRAND}'...")

docs = list(_db["records"].find(
    {"brand": BRAND, "ai_analysed": True},
    {
        "platform": 1, "record_type": 1, "source_title": 1,
        "text": 1, "captions_text": 1, "transcript_text": 1,
        "url": 1, "created_at": 1,
        "ai_overall_sentiment": 1,
        "ai_aspect_analysis": 1,
        "ai_suggestions": 1,
    }
))

if not docs:
    print("\nNo analysed records found in 'records' collection.")
    print("This means Cell 6 (Save to MongoDB) may not have run yet,")
    print("OR the records were not back-updated.")
    print("\nFalling back to session variables (df_sample)...")

    # Fallback: read from df_sample in session memory
    if "df_sample" in globals() and not df_sample.empty:
        analysed = df_sample.dropna(subset=["ai_analysis"]).copy()
        print(f"Found {len(analysed)} rows in df_sample with ai_analysis.")
        docs = []
        for _, row in analysed.iterrows():
            a = row.get("ai_analysis", {}) or {}
            docs.append({
                "_id":                  row.get("_id", ""),
                "platform":             row.get("platform", ""),
                "record_type":          row.get("record_type", ""),
                "source_title":         row.get("source_title", ""),
                "text":                 row.get("text", ""),
                "captions_text":        row.get("captions_text", ""),
                "transcript_text":      row.get("transcript_text", ""),
                "url":                  row.get("url", ""),
                "created_at":           row.get("created_at", ""),
                "ai_overall_sentiment": a.get("overall_sentiment", ""),
                "ai_aspect_analysis":   a.get("aspect_analysis", []),
                "ai_suggestions":       a.get("actionable_suggestions", []),
            })
    else:
        raise ValueError(
            "No data found in MongoDB OR session.\n"
            "Run the Analysis notebook (Cell 5) and Save notebook (Cell 6) first."
        )

print(f"Loaded {len(docs)} analysed records.\n")

# ── Build display DataFrame ───────────────────────────────────────────────────
def make_combined_text(doc):
    parts = [
        str(doc.get("source_title", "") or ""),
        str(doc.get("text",         "") or ""),
        str(doc.get("captions_text","") or ""),
        str(doc.get("transcript_text","") or ""),
    ]
    combined = " ".join(p.strip() for p in parts if p.strip())
    return combined[:600]  # show first 600 chars

def format_aspects(aspects):
    if not aspects: return "—"
    lines = []
    for a in aspects:
        if not isinstance(a, dict): continue
        sentiment = str(a.get("sentiment", "")).lower()
        emoji = "🟢" if sentiment == "positive" else "🔴" if sentiment == "negative" else "🔵"
        lines.append(f"{emoji} {a.get('aspect','?')} — {a.get('reason','')[:120]}")
    return "\n".join(lines)

def format_suggestions(suggs):
    if not suggs: return "—"
    return "\n".join(f"• {s[:120]}" for s in suggs if s)

def sentiment_emoji(s):
    s = str(s).lower()
    if s == "positive": return "🟢 positive"
    if s == "negative": return "🔴 negative"
    if s == "neutral":  return "🔵 neutral"
    return s or "—"

rows = []
for doc in docs:
    rows.append({
        "Platform":           doc.get("platform", ""),
        "Type":               doc.get("record_type", ""),
        "Original Text":      make_combined_text(doc),
        "Overall Sentiment":  sentiment_emoji(doc.get("ai_overall_sentiment", "")),
        "Aspects Found":      format_aspects(doc.get("ai_aspect_analysis", [])),
        "Suggestions":        format_suggestions(doc.get("ai_suggestions", [])),
        "URL":                str(doc.get("url", "") or "")[:80],
        "Date":               str(doc.get("created_at", "") or "")[:10],
    })

df_viewer = pd.DataFrame(rows)

# ── Summary stats ─────────────────────────────────────────────────────────────
print("=" * 60)
print(f"  ANALYSIS VIEWER — {BRAND}  ({len(df_viewer)} records)")
print("=" * 60)

total = len(df_viewer)
sentiment_counts = pd.Series([doc.get("ai_overall_sentiment","") for doc in docs]).value_counts()
print(f"\nOverall sentiment breakdown:")
for sent, count in sentiment_counts.items():
    emoji = "🟢" if sent == "positive" else "🔴" if sent == "negative" else "🔵"
    pct = 100 * count // total
    print(f"  {emoji} {sent:<12} {count:>4} records  ({pct}%)")

# Platform breakdown
print(f"\nPlatform breakdown:")
plat_counts = pd.Series([doc.get("platform","") for doc in docs]).value_counts()
for plat, count in plat_counts.items():
    print(f"  {plat:<15} {count:>4} records")

# Aspect counts
all_aspects_flat = []
for doc in docs:
    for a in (doc.get("ai_aspect_analysis") or []):
        if isinstance(a, dict) and a.get("aspect"):
            all_aspects_flat.append(a["aspect"])

print(f"\nMost identified aspects (across all {total} records):")
aspect_series = pd.Series(all_aspects_flat).value_counts().head(15)
for aspect, count in aspect_series.items():
    print(f"  {aspect:<30} {count:>4}x")

# ── OPTION A: Full interactive HTML table ────────────────────────────────────
print("\n\n" + "="*60)
print("  FULL RECORD-BY-RECORD VIEW (scroll right to see all columns)")
print("="*60)

pd.set_option("display.max_colwidth", 300)
pd.set_option("display.max_rows", 200)

def color_sentiment(val):
    if "positive" in str(val).lower(): return "background-color: #d4edda; color: #155724"
    if "negative" in str(val).lower(): return "background-color: #f8d7da; color: #721c24"
    if "neutral"  in str(val).lower(): return "background-color: #cce5ff; color: #004085"
    return ""

styled = (
    df_viewer.style
    .applymap(color_sentiment, subset=["Overall Sentiment"])
    .set_properties(**{"white-space": "pre-wrap", "text-align": "left",
                       "font-size": "12px", "border": "1px solid #dee2e6"})
    .set_table_styles([{
        "selector": "th",
        "props": [("background-color", "#343a40"), ("color", "white"),
                  ("font-size", "12px"), ("padding", "8px")]
    }])
)

display(styled)

# ── OPTION B: Filter by platform or sentiment ─────────────────────────────────
print("\n\n" + "="*60)
print("  FILTERED VIEWS")
print("="*60)

# Filter: only NEGATIVE records (most useful for accuracy checking)
print("\n🔴 NEGATIVE records only:")
neg_docs = [d for d in docs if str(d.get("ai_overall_sentiment","")).lower() == "negative"]
if neg_docs:
    for i, doc in enumerate(neg_docs[:20], 1):
        print(f"\n{'─'*55}")
        print(f"  [{i}] {doc.get('platform','').upper()} | {doc.get('record_type','')} | {doc.get('created_at','')[:10]}")
        print(f"  TEXT: {make_combined_text(doc)[:300]}")
        print(f"  ASPECTS:")
        for a in (doc.get("ai_aspect_analysis") or []):
            if isinstance(a, dict):
                print(f"    🔴 {a.get('aspect','?')} — {a.get('reason','')[:150]}")
        print(f"  SUGGESTIONS:")
        for s in (doc.get("ai_suggestions") or []):
            print(f"    • {str(s)[:150]}")
else:
    print("  No negative records found.")

# ── OPTION C: Export to CSV so you can review in Excel ───────────────────────
print("\n\n" + "="*60)
print("  EXPORT TO CSV (download and open in Excel to review)")
print("="*60)

# Flat CSV with one row per ASPECT (easiest to review accuracy)
flat_rows = []
for doc in docs:
    base_text = make_combined_text(doc)
    overall   = doc.get("ai_overall_sentiment", "")
    for a in (doc.get("ai_aspect_analysis") or []):
        if not isinstance(a, dict): continue
        flat_rows.append({
            "platform":          doc.get("platform", ""),
            "record_type":       doc.get("record_type", ""),
            "date":              str(doc.get("created_at",""))[:10],
            "original_text":     base_text,
            "overall_sentiment": overall,
            "aspect":            a.get("aspect", ""),
            "aspect_sentiment":  a.get("sentiment", ""),
            "reason":            a.get("reason", ""),
            "url":               doc.get("url", ""),
            # Leave blank for you to fill in manually
            "manual_correct_yn": "",
            "manual_notes":      "",
        })

df_export = pd.DataFrame(flat_rows)
csv_path  = f"/content/{BRAND}_analysis_review.csv"
df_export.to_csv(csv_path, index=False)

print(f"\nSaved: {csv_path}")
print(f"Total rows: {len(df_export)} (one row per aspect)")
print("\nTo download:")
print("  Files panel (left sidebar) → find the CSV → right-click → Download")
print("\nColumns in the CSV:")
print("  original_text     — what the customer actually said")
print("  overall_sentiment — what DeepSeek said (positive/negative/neutral)")
print("  aspect            — specific topic DeepSeek identified")
print("  aspect_sentiment  — DeepSeek's sentiment for that aspect")
print("  reason            — why DeepSeek said that")
print("  manual_correct_yn — YOU fill this in (yes/no)")
print("  manual_notes      — YOUR comments on accuracy")

# Download trigger
try:
    from google.colab import files
    files.download(csv_path)
    print("\nDownload triggered automatically.")
except Exception:
    print("\nManual download: Files panel → right-click the CSV → Download")


Loading analysed records for brand='KFC'...
Loaded 500 analysed records.

  ANALYSIS VIEWER — KFC  (500 records)

Overall sentiment breakdown:
  🔴 negative      229 records  (45%)
  🟢 positive      183 records  (36%)
  🔵 neutral        88 records  (17%)

Platform breakdown:
  youtube          349 records
  google_maps       95 records
  reddit            56 records

Most identified aspects (across all 500 records):
  Food Quality                    304x
  Other                           110x
  Staff Attitude                  104x
  Price                            51x
  Wait Time                        41x
  Promotions                       36x
  Cleanliness                      27x
  Service                          14x
  Overall Experience               11x
  Customer Service                 11x
  Delivery                         10x
  App Experience                    7x
  Location                          6x
  Packaging                         5x
  Portion Size                     

,Platform,Type,Original Text,Overall Sentiment,Aspects Found,Suggestions,URL,Date
0,reddit,post,"Any good burger places/restaurants on uber? Been having the same places as of recently. Chicken depot, street burger, fullr burger, holy dumpling, taco shack, all that kind of stuff. But i always seem to taste fatigue after I get halfway through the meal, and Kinda force myself to eat the rest cuz I dont wanna waste. Anybody got a good recommendation on a good place to get burgers/fried chicken for dinner, preferably not too expensive( also ones that arent KFC or burger king, The last time I had KFC stripes they were absurdly small and im not too confident in burger kind either) or any other r",🔴 negative,"🔴 Food Quality — The customer mentions 'taste fatigue' and having to force themselves to eat, and criticizes KFC strips as 'absurdly smal 🔴 Price — The customer wants places 'not too expensive', implying current options are pricey. 🔴 Variety — The customer expresses boredom with the same places and seeks new recommendations.",• Improve portion sizes and consistency of fried chicken strips to meet customer expectations. • Introduce new menu items or limited-time offers to combat taste fatigue and increase variety. • Consider value meal options or competitive pricing to attract cost-conscious customers.,https://www.reddit.com/r/srilanka/comments/1rlpadk/any_good_burger_placesrestaur,2026-03-05
1,reddit,comment,KFC chicken bucket tasted absolute sh@t Does anyone know a good KFC that puts spices? The recent ones Ive tried around Colombo is just batter fried chicken.,🔴 negative,🔴 Food Quality — Customer says chicken bucket tasted 'absolute sh@t' and describes it as 'just batter fried chicken' lacking spices.,"• Ensure consistent seasoning across all outlets, especially in Colombo. • Review and improve the spice blend recipe to meet customer expectations. • Conduct quality checks to prevent serving bland, unseasoned chicken.",https://www.reddit.com/r/srilanka/comments/1rg9y62/kfc_chicken_bucket_tasted_abs,2026-02-27
2,reddit,comment,"KFC chicken bucket tasted absolute sh@t So bad? some years ago I used to love KFC. But they started deteriorating I remember. First thing they did was get rid of their fantastic mashed potatoes. Then they started giving some stupid sauce with the biryani which prior to that I used to love because it was actual gravy. Yummy. And the chicken flavor was awesome. I loved it more KFC in other countries. Every time I come to this discussion forum I see complaints about so many things I used to love. Even Pizza Hut. It used to be awesome. Me and my friends always ordered Pizza Hut Pizza, Spaghetti B",🔴 negative,"🔴 Food Quality — Customer says 'KFC chicken bucket tasted absolute sh@t' and mentions deterioration in chicken flavor and removal of favo 🔴 Menu Changes — Customer complains about removal of 'fantastic mashed potatoes' and replacement of gravy with 'stupid sauce' in biryani. 🔴 Brand Consistency — Customer notes that KFC in other countries used to be better, implying decline in quality.",• Reintroduce popular discontinued items like mashed potatoes and original gravy to regain customer loyalty. • Improve chicken quality and consistency to match previous standards. • Monitor customer feedback on menu changes and consider reverting or improving new recipes.,https://www.reddit.com/r/srilanka/comments/1rg9y62/kfc_chicken_bucket_tasted_abs,2026-02-27
3,reddit,comment,KFC chicken bucket tasted absolute sh@t why do ppl goto KFC in 2026?,🔴 negative,🔴 Food Quality — Customer says chicken bucket tasted 'absolute sh@t'.,• Improve chicken quality and consistency across all outlets. • Investigate supply chain or cooking processes to ensure freshness. • Consider menu innovation to attract customers in 2026.,https://www.reddit.com/r/srilanka/comments/1rg9y62/kfc_chicken_bucket_tasted_abs,2026-02-28
4,reddit,post,"Best Place to eat around Negombo So since this is December I am thinking there might be lot offers in restaur



  FILTERED VIEWS

🔴 NEGATIVE records only:

───────────────────────────────────────────────────────
  [1] REDDIT | post | 2026-03-05
  TEXT: Any good burger places/restaurants on uber? Been having the same places as of recently. Chicken depot, street burger, fullr burger, holy dumpling, taco shack, all that kind of stuff. But i always seem to taste fatigue after I get halfway through the meal, and Kinda force myself to eat the rest cuz I
  ASPECTS:
    🔴 Food Quality — The customer mentions 'taste fatigue' and having to force themselves to eat, and criticizes KFC strips as 'absurdly small'.
    🔴 Price — The customer wants places 'not too expensive', implying current options are pricey.
    🔴 Variety — The customer expresses boredom with the same places and seeks new recommendations.
  SUGGESTIONS:
    • Improve portion sizes and consistency of fried chicken strips to meet customer expectations.
    • Introduce new menu items or limited-time offers to combat taste fatigue and increas

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Download triggered automatically.


In [ ]:
# Table 3 data
!pip install pymongo dnspython certifi
from pymongo import MongoClient
import certifi, pandas as pd

# =======================================================================
# LOAD SECRETS (same as analysis notebook — no re-entering)
# =======================================================================
try:
    from google.colab import userdata
    _MONGO_URI     = ""
    _MONGO_DB_NAME = "brandpulse"
    _BRAND         = "KFC"
    for name in ["MONGO_URI", "mongo_uri"]:
        try:
            v = (userdata.get(name) or "").strip()
            if v: _MONGO_URI = v; break
        except: pass
    for name in ["MONGO_DB_NAME", "mongo_db"]:
        try:
            v = (userdata.get(name) or "").strip()
            if v: _MONGO_DB_NAME = v; break
        except: pass
    for name in ["BRAND", "brand"]:
        try:
            v = (userdata.get(name) or "").strip()
            if v: _BRAND = v; break
        except: pass
except:
    import getpass
    _MONGO_URI     = getpass.getpass("MongoDB URI: ")
    _MONGO_DB_NAME = input("DB name [brandpulse]: ").strip() or "brandpulse"
    _BRAND         = input("Brand [KFC]: ").strip() or "KFC"

# Assign to global variables for use in the cell
MONGO_URI     = _MONGO_URI
MONGO_DB_NAME = _MONGO_DB_NAME
BRAND         = _BRAND

client = MongoClient(MONGO_URI, tls=True, tlsCAFile=certifi.where())
db = client["brandpulse"]

table3 = pd.DataFrame(list(db["records"].aggregate(
    [{"$match": {"brand": "KFC"}},
    {"$group": {"_id": {"platform": "$platform", "record_type": "$record_type"}, "count": {"$sum": 1}}},
    {"$sort": {"count": -1}}]
)))
print(table3)

# Table 4 data
table4 = pd.DataFrame(list(db["analysis_aspects"].find({"brand":"KFC"},
    {"aspect":1,"sentiment":1})))
print(table4.groupby(["aspect","sentiment"]).size().unstack(fill_value=0))

                                                    _id  count
0     {'platform': 'youtube', 'record_type': 'comment'}   1171
1  {'platform': 'google_maps', 'record_type': 'review'}    193
2      {'platform': 'reddit', 'record_type': 'comment'}     70
3       {'platform': 'youtube', 'record_type': 'video'}     39
4         {'platform': 'reddit', 'record_type': 'post'}     30
sentiment                              negative  neutral  positive
aspect                                                            
Accommodation                                 0        0         1
Ambiance                                      0        0         4
Ambience                                      0        0         1
Animal Welfare                                1        0         0
App Experience                                6        2         0
App Experience/Technology                     1        0         0
Atmosphere                                    3        0         0
Beverage Availabili